# CS781 Phase 7 — Prior Case Retrieval (PCR)

**Task:** for each query case, rank the candidates of **that split's own pool** and return exactly the **top 7** unique candidate IDs
(most → least relevant).

**Model:** siamese bi-encoder. Query and candidate documents → `MAX_*_CHUNKS` chunks of 256 tokens (spread across the document) →
Phase-1 backbone `encode()` → token pooling per chunk → mean/max/attention over chunks → projection (256-d) → L2 normalisation.
Score = cosine / τ. Loss = multi-positive InfoNCE over in-batch + sampled negatives **+ 0.3 ×** a pairwise hard-negative
ranking loss (softplus margin 0.1 against the 3 hardest negatives). Inference embeds the whole pool once and returns the
top 7 by cosine similarity (the query's own ID is excluded).

**Output (`prediction.json`):** a JSON **dictionary** query ID → 7 candidate IDs:

```json
{"query_001": ["candA", "candB", "candC", "candD", "candE", "candF", "candG"]}
```

**Primary metric:** Micro-F1@7 (secondary: Micro-Precision@7, Micro-Recall@7).

**Adaptation techniques**

| # | Key | Technique | Implementation |
|---|-----|-----------|----------------|
| 1 | `full`    | Full fine-tuning | every backbone + head parameter trainable |
| 2 | `partial` | Partial fine-tuning | last 25 % (≥ 1) of every `nn.ModuleList` of layers + head |
| 3 | `ia3`     | IA³ — learned rescaling vectors | `peft` — `IA3Config()` |
| 4 | `lorma`   | **LORMA (compulsory)** — low-rank **multiplicative** update `(I + s·BA)·W₀x` | Custom `LowRankMulLinear` injection |
| 5 | `kd`      | **Knowledge Distillation (compulsory)** | `DistillationExperiment` + listwise `listwise_kd_loss` (section 15) |

**How to run.** Set paths and run flags in section 2, then *Run All*. The default flags reproduce the best submission
(the KD student): preflight checks → train the teacher `exp_cfg` (decoder, full fine-tuning, lr 2e-5) → distil it into a
4-layer decoder student with LORMA → rank the test pool with the student → `submission.zip`.

| Stage | Section |
|---|---|
| Environment and imports | 1 |
| Paths and configuration | 2 |
| Data loading and validation | 3 |
| Tokenizer | 4 |
| Dataset and DataLoaders | 5 |
| Phase-1 backbone definitions | 6 |
| Backbone registry and checkpoint checks | 7 |
| Phase-7 model wrapper | 8 |
| Pooling | 9 |
| Adaptation methods | 10 |
| Experiment configuration | 11 |
| Training, evaluation and submission utilities | 12 |
| Preflight tests (optional, lightweight) | 13 |
| Main training experiment | 14 |
| Knowledge distillation (optional) | 15 |
| Test inference (ranking) | 16 |
| Submission validation and ZIP | 17 |

## 1. Environment and imports

In [ ]:
!pip install -q huggingface_hub safetensors peft

In [ ]:
import os, json, random, math, time, zipfile, copy, gc
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict, field, replace
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from tqdm.auto import tqdm

# Built-in PEFT support (LoRA, rsLoRA, IA3)
import peft
from peft import LoraConfig, IA3Config, get_peft_model

print("torch:", torch.__version__)
print("peft :", peft.__version__)

# ------------------------------------------------------------
# Determinism
# ------------------------------------------------------------
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)

## 2. Paths and configuration

All Kaggle paths and run flags live here. Nothing below hard-codes a checkpoint path.

In [ ]:
# ------------------------------------------------------------
# Phase-1 tokenizer and backbone checkpoints
# ------------------------------------------------------------
TOKENIZER_PATH     = "/kaggle/input/datasets/cs781ner/tokenizer7/legal_bpe.json"
DECODER_CHECKPOINT = "/kaggle/input/datasets/cs781ner/deconly7/pytorch_model.bin"
SSM_CHECKPOINT     = "/kaggle/input/datasets/sg123321/ssmonly/model.safetensors"
ENCDEC_CHECKPOINT  = "/kaggle/input/datasets/sg123321/converted-encdec-scratch"   # directory or file
CNN_CHECKPOINT     = "/kaggle/input/datasets/sg123321/cnn-atten/model.safetensors"

# Knowledge-distillation teacher (decoder + full fine-tuning). None = the checkpoint trained in section 14
# (teacher_cfg.run_dir / "best_model.pt"); or set the path of an attached checkpoint.
KD_TEACHER_CHECKPOINT = None

# Checkpoint used by test inference when the model was not trained in this session.
# None -> <run_dir>/best_model.pt of the selected configuration.
INFERENCE_CHECKPOINT = None

# ------------------------------------------------------------
# Chunking / loader settings
# ------------------------------------------------------------
CHUNK_TOKENS = 256
CHUNK_STRIDE = 192
MAX_QUERY_CHUNKS = 2
MAX_CAND_CHUNKS = 2
BATCH_SIZE = 1           # queries per batch
NUM_HARD_NEGATIVES = 3   # negatives sampled per query from its own split pool
NUM_WORKERS = 0          # 0: the shared TextChunkCache lives in the main process

# ------------------------------------------------------------
# What a "Run All" executes (defaults reproduce the best submission: KD student of the decoder full-FT teacher)
# ------------------------------------------------------------
RUN_PREFLIGHT = True                                      # section 13 (lightweight)
PREFLIGHT_BACKBONES = ["decoder", "encdec", "cnn", "ssm"]  # backbones checked in section 13
RUN_TINY_OVERFIT = False                                  # section 13
RUN_TRAINING = True                                       # section 14: trains exp_cfg (= the KD teacher)
RUN_KD = True                                             # section 15
RUN_INFERENCE = True                                      # sections 16-17
INFERENCE_SOURCE = "kd"                                   # "main" = exp_cfg model, "kd" = KD student

In [ ]:
# ============================================================
# HUGGING FACE SETUP
# ============================================================
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, hf_hub_download, list_repo_files

HF_TOKEN = UserSecretsClient().get_secret("hf_token_rp")
login(token=HF_TOKEN, add_to_git_credential=False)

# ============================================================
# PHASE 7 DATASET FROM HUGGING FACE
# ============================================================
DATA_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE_DIR = "Phase_7_PCR"

REPO_FILES = list_repo_files(DATA_REPO, repo_type="dataset", token=HF_TOKEN)
PHASE_FILES = [f for f in REPO_FILES if f.startswith(PHASE_DIR + "/")]
print(f"{PHASE_DIR} files in repo:")
for f in PHASE_FILES:
    print("  ", f)


def get_data_file(filename):
    """Download `filename`; if that exact path is absent, fall back to a UNIQUE basename match inside PHASE_DIR."""
    if filename not in REPO_FILES:
        base = filename.split("/")[-1]
        matches = [f for f in PHASE_FILES if f.split("/")[-1] == base]
        if len(matches) != 1:
            raise FileNotFoundError(f"'{filename}' not in repo; basename matches in {PHASE_DIR}: {matches}")
        print(f"[note] '{filename}' not found, using '{matches[0]}'")
        filename = matches[0]
    return Path(
        hf_hub_download(
            repo_id=DATA_REPO,
            filename=filename,
            repo_type="dataset",
            token=HF_TOKEN,
        )
    )


TRAIN_FILE = get_data_file(
    "Phase_7_PCR/Train/train.json"
)

TRAIN_CAND_FILE = get_data_file(
    "Phase_7_PCR/Train/train_candidates.json"
)

DEV_FILE = get_data_file(
    "Phase_7_PCR/Dev/dev.json"
)

DEV_CAND_FILE = get_data_file(
    "Phase_7_PCR/Dev/dev_candidates.json"
)

TEST_FILE = get_data_file(
    "Phase_7_PCR/Test/test.json"
)

TEST_CAND_FILE = get_data_file(
    "Phase_7_PCR/Test/test_candidates.json"
)

for name, p in [
    ("TRAIN_FILE", TRAIN_FILE),
    ("TRAIN_CAND_FILE", TRAIN_CAND_FILE),
    ("DEV_FILE", DEV_FILE),
    ("DEV_CAND_FILE", DEV_CAND_FILE),
    ("TEST_FILE", TEST_FILE),
    ("TEST_CAND_FILE", TEST_CAND_FILE),
]:
    print(f"{name:16s}: {p}")

## 3. Data loading and validation

Inspect the raw JSON first. Each split has its **own** query file and candidate pool:

| split | queries | candidate pool |
|-------|---------|----------------|
| train | `train.json` | `train_candidates.json` |
| dev   | `dev.json`   | `dev_candidates.json` |
| test  | `test.json`  | `test_candidates.json` |

Never mix pools across splits — test predictions must come **only** from `test_candidates.json`.

In [ ]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def describe_json(obj, name, depth=0, max_items=2):
    """Print the nested structure of a JSON object (types, sizes, first keys) to discover the schema."""
    pad = "  " * depth
    if isinstance(obj, dict):
        print(f"{pad}{name}: dict[{len(obj)}] keys={list(obj.keys())[:8]}")
        for k in list(obj.keys())[:max_items]:
            describe_json(obj[k], repr(k), depth + 1, max_items)
    elif isinstance(obj, list):
        print(f"{pad}{name}: list[{len(obj)}]")
        for i, x in enumerate(obj[:max_items]):
            describe_json(x, f"[{i}]", depth + 1, max_items)
    else:
        s = repr(obj)
        print(f"{pad}{name}: {s[:120]}{'...' if len(s) > 120 else ''}")


train_q_raw = load_json(TRAIN_FILE)
train_c_raw = load_json(TRAIN_CAND_FILE)
dev_q_raw   = load_json(DEV_FILE)
dev_c_raw   = load_json(DEV_CAND_FILE)
test_q_raw  = load_json(TEST_FILE)
test_c_raw  = load_json(TEST_CAND_FILE)

for name, obj in [("train", train_q_raw), ("train_candidates", train_c_raw),
                  ("dev", dev_q_raw), ("dev_candidates", dev_c_raw),
                  ("test", test_q_raw), ("test_candidates", test_c_raw)]:
    describe_json(obj, name)

In [ ]:
TOP_K = 7

def _get_text(item):
    txt = item["text"]
    if isinstance(txt, list):
        return " ".join(str(x) for x in txt if x)
    return str(txt)


def normalize_queries(raw, has_labels=True):
    out = []

    for qid, item in raw.items():
        q = {
            "id": str(qid),
            "text": _get_text(item)
        }

        if has_labels:
            q["relevant"] = [
                str(x) for x in item["relevant_candidates"]
            ]

        out.append(q)

    return out


def normalize_pool(raw):
    return {
        str(did): _get_text(item)
        for did, item in raw.items()
    }


train_queries = normalize_queries(train_q_raw)
dev_queries   = normalize_queries(dev_q_raw)
test_queries  = normalize_queries(test_q_raw, has_labels=False)

train_pool = normalize_pool(train_c_raw)
dev_pool   = normalize_pool(dev_c_raw)
test_pool  = normalize_pool(test_c_raw)

SPLITS = {
    "train": (train_queries, train_pool),
    "dev":   (dev_queries, dev_pool),
    "test":  (test_queries, test_pool),
}

for name, (qs, pool) in SPLITS.items():
    print(f"{name:5s}: {len(qs):5d} queries | {len(pool):6d} candidates")

In [ ]:
def check_split(name, queries, pool, has_labels=True):
    ids = [q["id"] for q in queries]
    assert len(ids) == len(set(ids)), f"{name}: duplicate query ids"
    n_self_in_pool = sum(q["id"] in pool for q in queries)
    print(f"{name}: queries whose own id is in the pool: {n_self_in_pool}")
    if has_labels:
        missing = [(q["id"], c) for q in queries for c in q["relevant"] if c not in pool]
        assert not missing, f"{name}: gold candidates missing from the pool, e.g. {missing[:5]}"
        n_gold = [len(q["relevant"]) for q in queries]
        print(f"{name}: gold/query mean {np.mean(n_gold):.2f} | min {min(n_gold)} | max {max(n_gold)} | "
              f"queries with 0 gold: {sum(n == 0 for n in n_gold)}")
    assert len(pool) >= TOP_K, f"{name}: pool smaller than {TOP_K}"


check_split("train", train_queries, train_pool)
check_split("dev", dev_queries, dev_pool)
check_split("test", test_queries, test_pool, has_labels=False)

# Additional checks: no query or candidate lost, query order preserved, test queries carry no labels.
for _name, (_qs, _pool), _raw_q, _raw_c in [("train", SPLITS["train"], train_q_raw, train_c_raw),
                                            ("dev", SPLITS["dev"], dev_q_raw, dev_c_raw),
                                            ("test", SPLITS["test"], test_q_raw, test_c_raw)]:
    assert [q["id"] for q in _qs] == [str(k) for k in _raw_q], f"{_name}: query IDs lost or reordered"
    assert set(_pool) == {str(k) for k in _raw_c}, f"{_name}: candidate IDs lost"
assert all("relevant" not in q for q in test_queries), "test queries must not carry labels"
print("Data validation passed.")

## 4. Tokenizer

Same `ByteLevelBPE` class and `legal_bpe.json` as Phase 3 (unchanged).

In [ ]:
#BPE tokenizer

def train_tokenizer(train_text, vocab_size):
    tokenizer = ByteLevelBPE(vocab_size=vocab_size)
    tokenizer.train([train_text])
    return tokenizer


import json
from collections import Counter, defaultdict
from tqdm.auto import tqdm


BPE_SPECIALS = [
    "<pad>",
    "<unk>",
    "<s>",
    "</s>"
]

# End-of-word marker.
# UTF-8 bytes are integers 0..255, so 256 cannot collide.
EOW = 256


class ByteLevelBPE:
    """
    Deterministic byte-level BPE tokenizer.

    - Initial units are UTF-8 bytes.
    - EOW marks the end of each word.
    - Case is preserved.
    - BPE never crosses word boundaries.
    - Ties are broken lexicographically.
    - Four special tokens are fixed at the beginning.
    """

    def __init__(self, vocab_size=5000):

        self.vocab_size = vocab_size
        self.merge_ranks = {}
        self.specials = list(BPE_SPECIALS)

        # List of learned merges.
        #
        # Each element is:
        #     (left_symbol, right_symbol)
        #
        self.merges = []

        # Vocabulary.
        self.itos = []
        self.stoi = {}

        # Internal merge rules.
        #
        # (left, right) -> merged_symbol
        #
        self.merge_rules = {}

        # Final representations of words seen during training.
        self.word_cache = {}
        self.token_to_symbol = {}

    # --------------------------------------------------------
    # Convert a byte/symbol tuple into a deterministic string
    # --------------------------------------------------------
    @staticmethod
    def string_to_symbol(token):
        # Standalone end-of-word marker
        if token == "</w>":
            return (EOW,)
    
        # Merged symbol ending with EOW
        if token.endswith("</w>>"):
            hex_part = token[3:-5]
            byte_part = tuple(
                int(hex_part[i:i+2], 16)
                for i in range(0, len(hex_part), 2)
            )
            return byte_part + (EOW,)
    
        # Ordinary byte or merged byte symbol
        hex_part = token[3:-1]
        return tuple(
            int(hex_part[i:i+2], 16)
            for i in range(0, len(hex_part), 2)
        )
    @staticmethod
    def symbol_to_string(symbol):

        # EOW is always the last symbol when present.
        if symbol and symbol[-1] == EOW:

            byte_part = symbol[:-1]

            if len(byte_part) == 0:
                return "</w>"

            hex_part = "".join(
                f"{b:02x}"
                for b in byte_part
            )

            return f"<0x{hex_part}</w>>"

        hex_part = "".join(
            f"{b:02x}"
            for b in symbol
        )

        return f"<0x{hex_part}>"
    def _build_merge_ranks(self):
        self.merge_ranks = {
            pair: rank
            for rank, pair in enumerate(self.merge_rules)
        }
    # --------------------------------------------------------
    # Deterministic ordering key for symbols
    # --------------------------------------------------------

    @staticmethod
    def symbol_key(symbol):
        return tuple(symbol)

    # --------------------------------------------------------
    # Convert a word into initial UTF-8 byte symbols
    # --------------------------------------------------------

    @staticmethod
    def word_to_symbols(word):

        byte_values = list(
            word.encode("utf-8")
        )

        symbols = [
            (b,)
            for b in byte_values
        ]

        # Explicit word boundary.
        symbols.append((EOW,))

        return symbols

    # --------------------------------------------------------
    # Count all pairs once initially
    # --------------------------------------------------------

    def _initial_pair_counts(
        self,
        word_symbols,
        word_counts
    ):

        pair_counts = Counter()

        # pair_words tells us which words contain a pair.
        pair_words = defaultdict(set)

        for word, symbols in word_symbols.items():

            freq = word_counts[word]

            for i in range(len(symbols) - 1):

                pair = (
                    symbols[i],
                    symbols[i + 1]
                )

                pair_counts[pair] += freq
                pair_words[pair].add(word)

        return pair_counts, pair_words

    # --------------------------------------------------------
    # Merge one pair in one word
    # --------------------------------------------------------

    @staticmethod
    def _merge_word(symbols, pair):

        left, right = pair

        new_symbol = left + right

        result = []

        i = 0

        while i < len(symbols):

            if (
                i + 1 < len(symbols)
                and symbols[i] == left
                and symbols[i + 1] == right
            ):

                result.append(new_symbol)

                i += 2

            else:

                result.append(symbols[i])

                i += 1

        return result

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    def train(self, texts):

        print("=" * 70)
        print("Training BYTE-LEVEL BPE")
        print("=" * 70)

        # ----------------------------------------------------
        # Count unique words.
        # ----------------------------------------------------

        word_counts = Counter()

        for text in texts:
            if not isinstance(text, str):
                continue

            for word in text.split():

                word_counts[word] += 1

        print(
            f"Unique words: {len(word_counts):,}"
        )

        # ----------------------------------------------------
        # Initial representation of each word.
        # ----------------------------------------------------

        word_symbols = {
            word: self.word_to_symbols(word)
            for word in word_counts
        }

        # ----------------------------------------------------
        # Initial vocabulary symbols.
        # ----------------------------------------------------

        initial_symbols = [
            (b,)
            for b in range(256)
        ]

        initial_symbols.append((EOW,))

        # Four specials + initial byte symbols.
        current_vocab_size = (
            len(BPE_SPECIALS)
            + len(initial_symbols)
        )

        print(
            f"Initial vocabulary size: "
            f"{current_vocab_size:,}"
        )

        if current_vocab_size > self.vocab_size:

            raise ValueError(
                "Initial byte vocabulary already exceeds "
                "the requested vocabulary size."
            )

        # ----------------------------------------------------
        # Count pairs once.
        # ----------------------------------------------------

        pair_counts, pair_words = \
            self._initial_pair_counts(
                word_symbols,
                word_counts
            )

        # ----------------------------------------------------
        # Main BPE loop.
        # ----------------------------------------------------

        merge_no = 0

        while current_vocab_size < self.vocab_size:

            if not pair_counts:
                break

            # Ignore pairs occurring only once.
            valid_pairs = [
                pair
                for pair, count in pair_counts.items()
                if count > 1
            ]

            if not valid_pairs:
                print(
                    "No pair occurs more than once. "
                    "Stopping early."
                )
                break

            # Highest frequency first.
            # Lexicographically smallest pair on ties.
            best_pair = min(
                valid_pairs,
                key=lambda pair: (
                    -pair_counts[pair],
                    self.symbol_key(pair[0]),
                    self.symbol_key(pair[1])
                )
            )

            best_count = pair_counts[best_pair]

            left, right = best_pair
            new_symbol = left + right

            # ------------------------------------------------
            # Record merge.
            # ------------------------------------------------

            self.merge_rules[best_pair] = new_symbol

            self.merges.append(
                (
                    self.symbol_to_string(left),
                    self.symbol_to_string(right)
                )
            )

            # ------------------------------------------------
            # Words affected by this merge.
            # ------------------------------------------------

            affected_words = list(
                pair_words.get(best_pair, set())
            )

            # Remove the selected pair's old contribution.
            del pair_counts[best_pair]

            pair_words.pop(
                best_pair,
                None
            )

            # ------------------------------------------------
            # Update each affected word.
            # ------------------------------------------------

            for word in affected_words:

                old_symbols = word_symbols[word]

                old_pairs = []

                for i in range(
                    len(old_symbols) - 1
                ):

                    old_pairs.append(
                        (
                            old_symbols[i],
                            old_symbols[i + 1]
                        )
                    )

                # Remove old pair contributions.
                freq = word_counts[word]

                for pair in old_pairs:

                    pair_counts[pair] -= freq

                    if pair_counts[pair] <= 0:
                        pair_counts.pop(
                            pair,
                            None
                        )

                    if pair in pair_words:
                        pair_words[pair].discard(word)

                        if not pair_words[pair]:
                            pair_words.pop(
                                pair,
                                None
                            )

                # Perform merge.
                new_symbols = self._merge_word(
                    old_symbols,
                    best_pair
                )

                word_symbols[word] = new_symbols

                # Add new pair contributions.
                new_pairs = []

                for i in range(
                    len(new_symbols) - 1
                ):

                    new_pairs.append(
                        (
                            new_symbols[i],
                            new_symbols[i + 1]
                        )
                    )

                for pair in new_pairs:

                    pair_counts[pair] += freq
                    pair_words[pair].add(word)

            current_vocab_size += 1
            merge_no += 1

            if merge_no % 250 == 0:

                print(
                    f"Merges: {merge_no:4d} | "
                    f"Vocab: {current_vocab_size:5d} | "
                    f"Best pair frequency: {best_count:,}"
                )

        # ----------------------------------------------------
        # Construct final vocabulary.
        #
        # First:
        #   <pad>, <unk>, <s>, </s>
        #
        # Then initial symbols.
        #
        # Then symbols introduced by merges in merge order.
        # ----------------------------------------------------

        merged_symbols = list(
            self.merge_rules.values()
        )

        self.itos = (
            list(BPE_SPECIALS)
            +
            [
                self.symbol_to_string(s)
                for s in initial_symbols
            ]
            +
            [
                self.symbol_to_string(s)
                for s in merged_symbols
            ]
        )

        self.token_to_symbol = {}

        for symbol in initial_symbols:
            token = self.symbol_to_string(symbol)
            self.token_to_symbol[token] = symbol

        for symbol in merged_symbols:
            token = self.symbol_to_string(symbol)
            self.token_to_symbol[token] = symbol

        # Remove duplicates while preserving order.
        final_itos = []
        seen = set()

        for token in self.itos:

            if token not in seen:

                seen.add(token)
                final_itos.append(token)

        self.itos = final_itos[
            :self.vocab_size
        ]

        self.stoi = {
            token: i
            for i, token in enumerate(self.itos)
        }

        # Save representations of training words.
        self.word_cache = word_symbols

        print(
            "\nTraining complete."
        )

        print(
            f"Final vocabulary size: "
            f"{len(self.itos):,}"
        )

        print(
            f"Number of merges: "
            f"{len(self.merges):,}"
        )
        self._build_merge_ranks()
        return self

    # --------------------------------------------------------
    # Tokenize a word
    # --------------------------------------------------------

    def tokenize_word(self, word):
        # Use cached representation for training words.
        if word in self.word_cache:
            symbols = self.word_cache[word]
        else:
            symbols = self.word_to_symbols(word)
    
            while len(symbols) > 1:
                # Find adjacent pairs that have learned merges.
                candidates = [
                    (self.merge_ranks[(symbols[i], symbols[i + 1])], i)
                    for i in range(len(symbols) - 1)
                    if (symbols[i], symbols[i + 1]) in self.merge_ranks
                ]
    
                if not candidates:
                    break
    
                # Apply the earliest learned merge.
                _, best_idx = min(candidates)
    
                pair = (symbols[best_idx], symbols[best_idx + 1])
                new_symbol = self.merge_rules[pair]
    
                # Merge all non-overlapping occurrences.
                symbols = self._merge_word(symbols, pair)
    
        return [
            self.symbol_to_string(symbol)
            for symbol in symbols
        ]

    # --------------------------------------------------------
    # Tokenize complete sentence
    # --------------------------------------------------------

    def tokenize(self, text):

        tokens = []

        for word in text.split():

            tokens.extend(
                self.tokenize_word(word)
            )

        return tokens

    def encode(self, text):
        tokens = self.tokenize(text)

        ids = []
        for token in tokens:
            if token in self.stoi:
                ids.append(self.stoi[token])
            else:
                # unknown token
                ids.append(self.stoi["<unk>"])

        return ids

    def decode(self, ids):
      byte_values = []

      for idx in ids:
          token = self.itos[idx]

          if token in BPE_SPECIALS:
              continue

          symbol = self.token_to_symbol[token]

          for b in symbol:
              if b == EOW:
                  byte_values.append(ord(" "))
              else:
                  byte_values.append(b)

      # Remove the final space
      if byte_values and byte_values[-1] == ord(" "):
          byte_values.pop()

      return bytes(byte_values).decode("utf-8")

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    
        tokenizer = cls(vocab_size=data["vocab_size"])
        tokenizer.merges = data["merges"]
        tokenizer.itos = data["itos"]
        tokenizer.stoi = {
            token: i for i, token in enumerate(tokenizer.itos)
        }
    
        # Initial symbols
        initial_symbols = [(b,) for b in range(256)]
        initial_symbols.append((EOW,))
    
        tokenizer.token_to_symbol = {
            tokenizer.symbol_to_string(symbol): symbol
            for symbol in initial_symbols
        }
    
        # Reconstruct merge rules in training order
        for left_str, right_str in tokenizer.merges:
            left = tokenizer.string_to_symbol(left_str)
            right = tokenizer.string_to_symbol(right_str)
            merged = left + right
    
            tokenizer.merge_rules[(left, right)] = merged
            merged_str = tokenizer.symbol_to_string(merged)
            tokenizer.token_to_symbol[merged_str] = merged
        tokenizer._build_merge_ranks()  
        return tokenizer

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    def save(self, path):

        data = {
            "merges": self.merges,
            "itos": self.itos,
            "vocab_size": self.vocab_size
        }

        with open(
            path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                data,
                f,
                ensure_ascii=False
            )

        print(
            f"Saved: {path}"
        )

In [ ]:
tokenizer = ByteLevelBPE.load(TOKENIZER_PATH)

print("Vocabulary size:", len(tokenizer.itos))
print("Number of merges:", len(tokenizer.merges))

PAD_ID = tokenizer.stoi["<pad>"]
UNK_ID = tokenizer.stoi["<unk>"]
BOS_ID = tokenizer.stoi["<s>"]
EOS_ID = tokenizer.stoi["</s>"]

print("PAD/UNK/BOS/EOS:", PAD_ID, UNK_ID, BOS_ID, EOS_ID)

_probe = "The appellant filed a petition under Section 9 of the Act."
_ids = tokenizer.encode(_probe)
print(len(_ids), _ids[:20])
print(repr(tokenizer.decode(_ids)))

## 5. Dataset and DataLoaders

Default formulation: **bi-encoder + contrastive learning** (query and candidates encoded separately, cosine similarity, InfoNCE with
in-batch negatives + hard negatives from the same pool). A cross-encoder (`[query ; candidate] → score`) is the alternative — keep the same
output contract (`scores [B, M]`) so the training loop and KD stay unchanged.

**Batch contract (bi-encoder)** expected by the training loops (sections 12 & 15) and the model (section 8):

| key | shape | notes |
|-----|-------|-------|
| `query_input_ids` | `[B, Cq, L]` long | query chunks, `PAD_ID` padded |
| `query_attention_mask` | `[B, Cq, L]` long | |
| `query_chunk_mask` | `[B, Cq]` bool | 1 = real chunk |
| `cand_input_ids` | `[M, Cc, L]` long | all candidates in the batch (positives + hard negatives, de-duplicated) |
| `cand_attention_mask` | `[M, Cc, L]` long | |
| `cand_chunk_mask` | `[M, Cc]` bool | |
| `relevance` | `[B, M]` float | 1 if candidate m is gold for query b (multiple positives allowed), else 0 |
| `query_ids` / `cand_ids` | `List[str]` | not moved to device |

Every non-gold candidate in the batch acts as a negative for each query (**in-batch negatives**).

Chunk selection: when a document has more chunks than allowed, chunks evenly spaced across the whole document are kept.

In [ ]:
def chunk_token_ids(ids, chunk_len, stride, max_chunks=None, add_bos_eos=True):
    """
    Generic sliding-window chunker (no task logic).
      ids        : token ids of one full document (no BOS/EOS)
      chunk_len  : tokens per chunk INCLUDING BOS/EOS when add_bos_eos=True
                   (must be <= BACKBONE_MAX_TOKENS[backbone] — 1024 for encdec)
      stride     : step between chunk starts (stride < body length => overlap)
      max_chunks : keep at most this many chunks (None = all). Selection policy
                   (first-N / evenly spaced / head+tail) is a Phase-specific choice.
    Returns List[List[int]] (always >= 1 chunk, even for an empty document).
    """
    body = chunk_len - 2 if add_bos_eos else chunk_len
    assert body > 0 and 0 < stride <= body, (chunk_len, stride)
    chunks = []
    start = 0
    while True:
        piece = ids[start:start + body]
        chunks.append([BOS_ID] + piece + [EOS_ID] if add_bos_eos else piece)
        if start + body >= len(ids):
            break
        start += stride
    if max_chunks is not None:
        chunks = chunks[:max_chunks]
    return chunks


print(chunk_token_ids(list(range(10, 25)), chunk_len=8, stride=4))

In [ ]:
class TextChunkCache:
    """Tokenize + chunk each document ONCE (queries and candidates are reused across many batches)."""

    def __init__(self, tokenizer, chunk_tokens=CHUNK_TOKENS, chunk_stride=CHUNK_STRIDE):
        self.tokenizer = tokenizer
        self.chunk_tokens = chunk_tokens
        self.chunk_stride = chunk_stride
        self._cache = {}

    def get(self, key, text, max_chunks):
        """key: unique id (e.g. ("pool", "dev", cand_id)); returns List[List[int]] with <= max_chunks chunks."""
        if key not in self._cache:
            ids = self.tokenizer.encode(text)
            self._cache[key] = chunk_token_ids(ids, self.chunk_tokens, self.chunk_stride, max_chunks=None)
        return self._cache[key][:max_chunks]   # chunk-selection policy is a Phase-7 choice


chunk_cache = TextChunkCache(tokenizer)



class PCRDataset(Dataset):
    """
    One item = one query with its positive cases and sampled negatives.
    Candidate documents always come from the query's own split pool.
    """

    def __init__(
        self,
        queries,
        pool,
        split_name,
        cache=chunk_cache,
        num_hard_negatives=NUM_HARD_NEGATIVES,
        max_positives=None,
        seed=SEED,
    ):
        self.pool = pool
        self.split_name = split_name
        self.cache = cache
        self.num_hard_negatives = num_hard_negatives
        self.max_positives = max_positives
        self.rng = random.Random(seed)

        # Queries with no valid positives cannot contribute to
        # multi-positive contrastive learning.
        self.queries = []

        for q in queries:
            qid = str(q["id"])

            positives = [
                str(pid)
                for pid in q.get("relevant", [])
                if str(pid) in pool and str(pid) != qid
            ]

            if positives:
                self.queries.append({
                    "id": qid,
                    "text": q["text"],
                    "relevant": positives,
                })

        # Keep the candidate pool IDs as strings.
        self.pool_ids = list(pool.keys())

    def __len__(self):
        return len(self.queries)

    def _select_chunks(self, chunks, max_chunks):
        """
        Select chunks across the entire document rather than
        always taking the first max_chunks.
        """
        if len(chunks) <= max_chunks:
            return chunks

        if max_chunks == 1:
            return [chunks[len(chunks) // 2]]

        indices = [
            round(i * (len(chunks) - 1) / (max_chunks - 1))
            for i in range(max_chunks)
        ]

        return [chunks[i] for i in indices]

    def _get_chunks(self, key, text, max_chunks):
        # Retrieve all cached chunks before applying selection.
        chunks = self.cache.get(key, text, max_chunks=None)

        return self._select_chunks(chunks, max_chunks)

    def __getitem__(self, idx):
        q = self.queries[idx]

        qid = q["id"]
        positive_ids = list(q["relevant"])

        if self.max_positives is not None:
            if len(positive_ids) > self.max_positives:
                positive_ids = self.rng.sample(
                    positive_ids,
                    self.max_positives
                )

        positive_set = set(positive_ids)

        # Sample negatives from the same split's pool.
        negative_candidates = [
            did for did in self.pool_ids
            if did not in positive_set and did != qid
        ]

        n_neg = min(
            self.num_hard_negatives,
            len(negative_candidates)
        )

        negative_ids = self.rng.sample(
            negative_candidates,
            n_neg
        )

        cand_ids = positive_ids + negative_ids

        query_chunks = self._get_chunks(
            (self.split_name, "query", qid),
            q["text"],
            MAX_QUERY_CHUNKS,
        )

        cand_chunks = [
            self._get_chunks(
                (self.split_name, "pool", cid),
                self.pool[cid],
                MAX_CAND_CHUNKS,
            )
            for cid in cand_ids
        ]

        return {
            "query_id": qid,
            "query_chunks": query_chunks,
            "cand_ids": cand_ids,
            "cand_chunks": cand_chunks,
            "positive_ids": set(positive_ids),
        }


def pcr_collator(batch):
    """
    Deduplicate candidates across the batch.

    Returns:
        query_input_ids       [B, Cq, L]
        query_attention_mask  [B, Cq, L]
        query_chunk_mask      [B, Cq]

        cand_input_ids        [M, Cc, L]
        cand_attention_mask   [M, Cc, L]
        cand_chunk_mask       [M, Cc]

        relevance             [B, M]
    """

    batch_size = len(batch)

    # --------------------------------------------------
    # 1. Deduplicate candidates across the batch
    # --------------------------------------------------
    unique_cand_ids = []
    cand_id_to_idx = {}

    for item in batch:
        for cid in item["cand_ids"]:
            if cid not in cand_id_to_idx:
                cand_id_to_idx[cid] = len(unique_cand_ids)
                unique_cand_ids.append(cid)

    # --------------------------------------------------
    # 2. Construct multi-positive relevance matrix
    # --------------------------------------------------
    relevance = torch.zeros(
        batch_size,
        len(unique_cand_ids),
        dtype=torch.float32,
    )

    for b, item in enumerate(batch):
        for pid in item["positive_ids"]:
            if pid in cand_id_to_idx:
                relevance[b, cand_id_to_idx[pid]] = 1.0

    # --------------------------------------------------
    # 3. Collect query and candidate chunk sequences
    # --------------------------------------------------
    query_chunk_lists = [
        item["query_chunks"]
        for item in batch
    ]

    cand_chunk_lists = {}

    for item in batch:
        for cid, chunks in zip(
            item["cand_ids"],
            item["cand_chunks"]
        ):
            if cid not in cand_chunk_lists:
                cand_chunk_lists[cid] = chunks

    unique_cand_chunk_lists = [
        cand_chunk_lists[cid]
        for cid in unique_cand_ids
    ]

    # --------------------------------------------------
    # 4. Padding helper
    # --------------------------------------------------
    def pad_documents(documents):
        """
        documents: List[List[List[int]]]
                   [documents, chunks, token_ids]

        Returns:
            input_ids      [N, C, L]
            attention_mask [N, C, L]
            chunk_mask     [N, C]
        """

        n_docs = len(documents)

        max_chunks = max(
            len(chunks) for chunks in documents
        )

        max_tokens = max(
            len(chunk)
            for chunks in documents
            for chunk in chunks
        )

        input_ids = torch.full(
            (n_docs, max_chunks, max_tokens),
            PAD_ID,
            dtype=torch.long,
        )

        attention_mask = torch.zeros(
            n_docs,
            max_chunks,
            max_tokens,
            dtype=torch.long,
        )

        chunk_mask = torch.zeros(
            n_docs,
            max_chunks,
            dtype=torch.float32,
        )

        for d, chunks in enumerate(documents):
            for c, chunk in enumerate(chunks):
                length = len(chunk)

                input_ids[d, c, :length] = torch.tensor(
                    chunk,
                    dtype=torch.long,
                )

                attention_mask[d, c, :length] = 1
                chunk_mask[d, c] = 1.0

        # Padded chunk slots receive one BOS token.
        # attention_mask = 1, chunk_mask = 0.
        padded_chunk_mask = chunk_mask == 0

        if padded_chunk_mask.any():
            input_ids[
                padded_chunk_mask, 0
            ] = BOS_ID

            attention_mask[
                padded_chunk_mask, 0
            ] = 1

        return input_ids, attention_mask, chunk_mask

    # --------------------------------------------------
    # 5. Pad query and candidate documents separately
    # --------------------------------------------------
    (
        query_input_ids,
        query_attention_mask,
        query_chunk_mask,
    ) = pad_documents(query_chunk_lists)

    (
        cand_input_ids,
        cand_attention_mask,
        cand_chunk_mask,
    ) = pad_documents(unique_cand_chunk_lists)

    # --------------------------------------------------
    # 6. Return batch contract
    # --------------------------------------------------
    return {
        "query_input_ids": query_input_ids,
        "query_attention_mask": query_attention_mask,
        "query_chunk_mask": query_chunk_mask,

        "cand_input_ids": cand_input_ids,
        "cand_attention_mask": cand_attention_mask,
        "cand_chunk_mask": cand_chunk_mask,

        "relevance": relevance,

        # Metadata stays outside the model's tensor inputs.
        "query_ids": [item["query_id"] for item in batch],
        "candidate_ids": unique_cand_ids,
    }

In [ ]:
# ============================================================
# DataLoaders (boilerplate)
# Dev/test RANKING uses run_inference() over the full pool (Section 12);
# the dev loader below only measures the contrastive loss.
# ============================================================

def build_dataloaders(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS):
    # num_workers=0: the shared TextChunkCache lives in the main process
    train_ds = PCRDataset(train_queries, train_pool, "train")
    dev_ds   = PCRDataset(dev_queries, dev_pool, "dev")

    g = torch.Generator()
    g.manual_seed(SEED)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              collate_fn=pcr_collator, num_workers=num_workers, generator=g)
    dev_loader = DataLoader(dev_ds, batch_size=batch_size, shuffle=False,
                            collate_fn=pcr_collator, num_workers=num_workers)
    return train_ds, dev_ds, train_loader, dev_loader


train_ds, dev_ds, train_loader, dev_loader = build_dataloaders()

print("Train items:", len(train_ds))
print("Dev items  :", len(dev_ds))

_b = next(iter(train_loader))
for k, v in _b.items():
    print(f"{k:22s}", tuple(v.shape) if torch.is_tensor(v) else type(v).__name__)

## 6. Phase-1 backbone definitions

Every backbone exposes `encode(input_ids, attention_mask=None) -> [B, L, D]`; the bi-encoder calls it on chunks.
Checkpoint paths are set in section 2.

### 6A. Decoder-only Transformer

In [ ]:
import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class DecoderConfig:
    vocab_size: int
    context_length: int
    n_embd: int
    n_layer: int
    n_head: int
    embd_pdrop: float = 0.1
    resid_pdrop: float = 0.1
    attn_pdrop: float = 0.1
    layer_norm_epsilon: float = 1e-5
    initializer_range: float = 0.02
    bos_token_id: int = 2
    eos_token_id: int = 3
    pad_token_id: int = 0
    n_inner: int | None = None
    activation_function: str = "gelu_new"
    scale_attn_weights: bool = True
    scale_attn_by_inverse_layer_idx: bool = False
    tie_word_embeddings: bool = True

    @property
    def feed_forward_size(self):
        return self.n_inner if self.n_inner is not None else 4 * self.n_embd


class CausalSelfAttention(nn.Module):
    """Causal multi-head self-attention with a fused QKV projection."""

    def __init__(self, config: DecoderConfig, layer_idx: int):
        super().__init__()
        assert config.n_embd % config.n_head == 0, (
            "n_embd must be divisible by n_head"
        )
        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head
        self.layer_idx = layer_idx
        self.scale_attn_weights = config.scale_attn_weights
        self.scale_attn_by_inverse_layer_idx = (
            config.scale_attn_by_inverse_layer_idx
        )

        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd, bias=True)
        self.out_proj = nn.Linear(config.n_embd, config.n_embd, bias=True)
        self.attn_dropout = nn.Dropout(config.attn_pdrop)
        self.resid_dropout = nn.Dropout(config.resid_pdrop)

    def forward(self, hidden_states, attention_mask=None):
            batch_size, seq_len, hidden_size = hidden_states.shape
        
            qkv = self.qkv(hidden_states)
            query, key, value = qkv.split(hidden_size, dim=-1)
        
            def split_heads(tensor):
                return tensor.view(
                    batch_size,
                    seq_len,
                    self.n_head,
                    self.head_dim
                ).transpose(1, 2)
        
            query = split_heads(query)
            key = split_heads(key)
            value = split_heads(value)
        
            scale = None
        
            if self.scale_attn_weights:
                scale = 1.0 / math.sqrt(self.head_dim)
        
            if self.scale_attn_by_inverse_layer_idx:
                if scale is None:
                    scale = 1.0
                scale = scale / float(self.layer_idx + 1)
        
            attn_mask = None
            is_causal = True
        
            if attention_mask is not None:
                key_is_valid = attention_mask.to(
                    device=hidden_states.device,
                    dtype=torch.bool
                )
        
                # [B, 1, 1, L]
                attn_mask = key_is_valid[:, None, None, :]
        
                # Since an explicit mask is supplied, combine causal mask manually
                causal = torch.ones(
                    seq_len,
                    seq_len,
                    dtype=torch.bool,
                    device=hidden_states.device
                ).tril()
        
                attn_mask = attn_mask & causal[None, None, :, :]
                is_causal = False
        
            attended = F.scaled_dot_product_attention(
                query,
                key,
                value,
                attn_mask=attn_mask,
                dropout_p=self.attn_dropout.p if self.training else 0.0,
                is_causal=is_causal,
                scale=scale,
            )
        
            attended = attended.transpose(1, 2).contiguous().view(
                batch_size,
                seq_len,
                hidden_size
            )
        
            return self.resid_dropout(
                self.out_proj(attended)
            )


class DecoderOnlyMLP(nn.Module):
    """Position-wise feed-forward block (expand -> activation -> project)."""

    def __init__(self, config: DecoderConfig):
        super().__init__()
        self.fc_in = nn.Linear(
            config.n_embd, config.feed_forward_size, bias=True
        )
        self.fc_out = nn.Linear(
            config.feed_forward_size, config.n_embd, bias=True
        )
        self.dropout = nn.Dropout(config.resid_pdrop)
        self.activation_name = config.activation_function

    def _activation(self, x):
        if self.activation_name == "gelu_new":
            # Tanh approximation of GELU.
            return 0.5 * x * (
                1.0
                + torch.tanh(
                    math.sqrt(2.0 / math.pi)
                    * (x + 0.044715 * torch.pow(x, 3.0))
                )
            )
        if self.activation_name == "gelu":
            return F.gelu(x)
        if self.activation_name == "relu":
            return F.relu(x)
        if self.activation_name in ("silu", "swish"):
            return F.silu(x)
        raise ValueError(
            f"Unsupported activation: {self.activation_name}"
        )

    def forward(self, hidden_states):
        return self.dropout(self.fc_out(self._activation(self.fc_in(hidden_states))))


class DecoderOnlyBlock(nn.Module):
    """Pre-norm Transformer block with residual attention and MLP paths."""

    def __init__(self, config: DecoderConfig, layer_idx: int):
        super().__init__()
        self.ln_1 = nn.LayerNorm(
            config.n_embd, eps=config.layer_norm_epsilon
        )
        self.attn = CausalSelfAttention(config, layer_idx)
        self.ln_2 = nn.LayerNorm(
            config.n_embd, eps=config.layer_norm_epsilon
        )
        self.mlp = DecoderOnlyMLP(config)

    def forward(self, hidden_states, attention_mask=None):
        hidden_states = hidden_states + self.attn(
            self.ln_1(hidden_states), attention_mask=attention_mask
        )
        hidden_states = hidden_states + self.mlp(self.ln_2(hidden_states))
        return hidden_states


class DecoderOnlyTransformer(nn.Module):
    """Decoder-only causal language model implemented in raw PyTorch."""

    def __init__(self, config: DecoderConfig):
        super().__init__()
        if config.n_embd % config.n_head != 0:
            raise ValueError("n_embd must be divisible by n_head")
        if config.context_length <= 0:
            raise ValueError("context_length must be positive")
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding = nn.Embedding(
            config.context_length, config.n_embd
        )
        self.embedding_dropout = nn.Dropout(config.embd_pdrop)
        self.blocks = nn.ModuleList(
            DecoderOnlyBlock(config, layer_idx)
            for layer_idx in range(config.n_layer)
        )
        self.final_ln = nn.LayerNorm(
            config.n_embd, eps=config.layer_norm_epsilon
        )
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        self.apply(self._initialize_weights)
        if config.tie_word_embeddings:
            # Weight tying: the output projection shares the token-embedding matrix.
            self.lm_head.weight = self.token_embedding.weight

    def _initialize_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def _validate_inputs(self, input_ids, attention_mask=None):
        if input_ids.ndim != 2:
            raise ValueError("input_ids must have shape [batch, sequence]")
        batch_size, seq_len = input_ids.shape
        if seq_len == 0:
            raise ValueError("input sequence must not be empty")
        assert seq_len <= self.config.context_length, (
            f"sequence length {seq_len} exceeds context length "
            f"{self.config.context_length}"
        )
        if attention_mask is not None:
            if attention_mask.shape != (batch_size, seq_len):
                raise ValueError(
                    "attention_mask must have shape [batch, sequence]"
                )
        return batch_size, seq_len

    def _embed(self, input_ids):
        _, seq_len = input_ids.shape
        position_ids = torch.arange(
            seq_len, dtype=torch.long, device=input_ids.device
        ).unsqueeze(0)
        return self.embedding_dropout(
            self.token_embedding(input_ids)
            + self.position_embedding(position_ids)
        )

    def encode(self, input_ids, attention_mask=None):
        """Return final normalized hidden states with shape [B, L, D]."""
        self._validate_inputs(input_ids, attention_mask)
        hidden_states = self._embed(input_ids)
        for block in self.blocks:
            hidden_states = block(hidden_states, attention_mask)
        return self.final_ln(hidden_states)

    def debug_hidden_states(self, input_ids, attention_mask=None):
        """Expose intermediate states for optional migration debugging."""
        self._validate_inputs(input_ids, attention_mask)
        hidden_states = self._embed(input_ids)
        states = {"embeddings": hidden_states}
        for layer_idx, block in enumerate(self.blocks):
            hidden_states = block(hidden_states, attention_mask)
            states[f"block_{layer_idx}"] = hidden_states
        states["final_norm"] = self.final_ln(hidden_states)
        return states

    def forward(self, input_ids, attention_mask=None, labels=None):
        hidden_states = self.encode(input_ids, attention_mask)
        logits = self.lm_head(hidden_states)
        loss = None

        if labels is not None:
            if labels.shape != input_ids.shape:
                raise ValueError("labels must have the same shape as input_ids")
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            valid_labels = shift_labels.ne(-100)
            if attention_mask is not None:
                valid_labels = valid_labels & attention_mask[:, 1:].bool()
            
            shift_labels = shift_labels.masked_fill(~valid_labels, -100)
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )

        return {
            "loss": loss,
            "logits": logits,
            "hidden_states": hidden_states,
        }

In [ ]:
decoder_config = DecoderConfig(
    vocab_size=16000,
    context_length=2048,
    n_embd=640,
    n_layer=8,
    n_head=8,
    embd_pdrop=0.1,
    resid_pdrop=0.1,
    attn_pdrop=0.1,
    layer_norm_epsilon=1e-5,
    initializer_range=0.02,
    bos_token_id=BOS_ID,
    eos_token_id=EOS_ID,
    pad_token_id=PAD_ID,
    n_inner=None,
    activation_function="gelu_new",
    scale_attn_weights=True,
    scale_attn_by_inverse_layer_idx=False,
    tie_word_embeddings=True,
)

### 6B. Diagonal SSM / S4D-inspired LM

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class SSMLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        dim,
        n_layer,
        max_seq_len,
        state_dim=16,
        expand=3,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.dim = dim
        self.n_layer = n_layer
        self.max_seq_len = max_seq_len

        self.embedding = nn.Embedding(
            vocab_size,
            dim
        )

        self.layers = nn.ModuleList([
            SSMBlock(
                dim=dim,
                state_dim=state_dim,
                expand=expand,
            )
            for _ in range(n_layer)
        ])

        self.norm_f = nn.LayerNorm(dim)

        self.lm_head = nn.Linear(
            dim,
            vocab_size,
            bias=False
        )

    def forward(
        self,
        input_ids,
        labels=None,
        **kwargs
    ):
        B, T = input_ids.shape

        if T > self.max_seq_len:
            raise ValueError(
                f"Sequence length {T} exceeds "
                f"max_seq_len={self.max_seq_len}"
            )

        x = self.embedding(input_ids)

        for layer in self.layers:
            x = layer(x)

        x = self.norm_f(x)

        logits = self.lm_head(x)

        if labels is not None:

            targets = torch.full_like(
                labels,
                -100
            )

            targets[:, :-1] = labels[:, 1:]

            loss = F.cross_entropy(
                logits.reshape(
                    -1,
                    self.vocab_size
                ),
                targets.reshape(-1),
                ignore_index=-100,
            )

            return {
                "loss": loss
            }

        return {
            "logits": logits
        }
class SSMBlock(nn.Module):
    def __init__(
        self,
        dim,
        state_dim,
        expand=3,
    ):
        super().__init__()

        self.norm = nn.LayerNorm(dim)

        self.ssm = DiagonalSSM(
            dim=dim,
            state_dim=state_dim,
            expand=expand,
        )

    def forward(self, x):
        return x + self.ssm(self.norm(x))
class DiagonalSSM(nn.Module):
    """
    S4D-inspired diagonal state-space layer.

    Input : [B, T, C]
    Output: [B, T, C]
    """

    def __init__(
        self,
        dim,
        state_dim,
        expand=3,
    ):
        super().__init__()

        self.dim = dim
        self.state_dim = state_dim
        self.inner_dim = dim * expand

        # content + gate
        self.in_proj = nn.Linear(
            dim,
            2 * self.inner_dim
        )

        # Stable diagonal state transition
        # A will become negative
        A = torch.arange(
            1,
            state_dim + 1,
            dtype=torch.float32
        )

        self.A_log = nn.Parameter(
            torch.log(A)
        )

        # Per-channel SSM input/output weights
        self.B = nn.Parameter(
            torch.randn(
                self.inner_dim,
                state_dim
            ) * 0.02
        )

        self.C = nn.Parameter(
            torch.randn(
                self.inner_dim,
                state_dim
            ) * 0.02
        )

        # Per-channel time scale
        self.log_dt = nn.Parameter(
            torch.zeros(self.inner_dim)
        )

        # Direct skip term
        self.D = nn.Parameter(
            torch.ones(self.inner_dim)
        )

        self.out_proj = nn.Linear(
            self.inner_dim,
            dim
        )

    def forward(self, x):
        Bsz, T, _ = x.shape

        xz = self.in_proj(x)

        u, z = xz.chunk(
            2,
            dim=-1
        )

        # -------------------------------------------------
        # Step 1
        #
        # Construct stable continuous transition:
        #
        # A = -exp(A_log)
        #
        # shape:
        # [state_dim]
        # -------------------------------------------------

        A = -torch.exp(self.A_log)

        # -------------------------------------------------
        # Step 2
        #
        # dt must always be positive.
        #
        # use softplus(log_dt)
        #
        # shape:
        # [inner_dim]
        # -------------------------------------------------

        dt = F.softplus(self.log_dt)

        # -------------------------------------------------
        # Step 3
        #
        # Create discrete transition:
        #
        # A_bar[c,n] = exp(dt[c] * A[n])
        #
        # expected:
        # [inner_dim, state_dim]
        # -------------------------------------------------

        A_bar = torch.exp(
            dt.unsqueeze(-1) * A.unsqueeze(0)
        )

        # -------------------------------------------------
        # Step 4
        #
        # Zero-order-hold style input coefficient:
        #
        # B_bar[c,n]
        #   = ((A_bar[c,n] - 1) / A[n]) * B[c,n]
        #
        # expected:
        # [inner_dim, state_dim]
        # -------------------------------------------------

        xA = dt.unsqueeze(-1) * A.unsqueeze(0)

        B_bar = (
            torch.expm1(xA) / A.unsqueeze(0)
        ) * self.B

        # -------------------------------------------------
        # Step 5
        #
        # Build time indices:
        #
        # t = [0, 1, ..., T-1]
        #
        # Then construct the impulse-response kernel:
        #
        # K[c,t] =
        #   sum_n C[c,n]
        #         * A_bar[c,n]^t
        #         * B_bar[c,n]
        #
        # Target shape:
        # [inner_dim, T]
        #
        # Hint:
        # avoid a Python loop over T.
        # Broadcasting / einsum should be used.
        # -------------------------------------------------

        t = torch.arange(
            T,
            device=x.device,
            dtype=A_bar.dtype
        )

        powers = A_bar.unsqueeze(-1) ** t
        # [inner_dim, state_dim, T]

        kernel = torch.einsum(
            "cn,cnt->ct",
            self.C * B_bar,
            powers
        )
        # [inner_dim, T]

        # -------------------------------------------------
        # Step 6 — FFT causal convolution
        #
        # u currently:
        # [B, T, inner_dim]
        #
        # transpose to:
        # [B, inner_dim, T]
        #
        # Use an FFT length >= 2*T.
        #
        # rfft(input)
        # rfft(kernel)
        # multiply
        # irfft
        #
        # finally truncate to first T values.
        #
        # This gives causal convolution:
        #
        # y[t] = sum_{i=0}^{t} K[i] u[t-i]
        # -------------------------------------------------

        u = u.transpose(1, 2)
        # [B, inner_dim, T]

        fft_len = 2 * T

        u_fft = u.float()
        kernel_fft = kernel.float()
        
        u_f = torch.fft.rfft(
            u_fft,
            n=fft_len
        )
        
        k_f = torch.fft.rfft(
            kernel_fft,
            n=fft_len
        )
        
        y = torch.fft.irfft(
            u_f * k_f.unsqueeze(0),
            n=fft_len
        )[..., :T]
        
        y = y.to(u.dtype)

        # -------------------------------------------------
        # Step 7
        #
        # Add direct SSM skip:
        #
        # y += D * u
        # -------------------------------------------------

        y = y + self.D.view(1, -1, 1) * u

        # restore:
        # [B, T, inner_dim]

        y = y.transpose(1, 2)

        # same gating idea as before
        y = y * F.silu(z)

        return self.out_proj(y)

In [ ]:
def ssm_encode(self, input_ids, attention_mask=None):
    B, T = input_ids.shape

    if T > self.max_seq_len:
        raise ValueError(
            f"Sequence length {T} exceeds max_seq_len={self.max_seq_len}"
        )

    x = self.embedding(input_ids)

    for layer in self.layers:
        x = layer(x)

    x = self.norm_f(x)

    return x

SSMLM.encode = ssm_encode

In [ ]:
SSM_DIM = 700
SSM_N_LAYER = 10
SSM_MAX_SEQ_LEN = 2048
SSM_STATE_DIM = 16
SSM_EXPAND = 3


def build_ssm_backbone():
    return SSMLM(
        vocab_size=16000,
        dim=SSM_DIM,
        n_layer=SSM_N_LAYER,
        max_seq_len=SSM_MAX_SEQ_LEN,
        state_dim=SSM_STATE_DIM,
        expand=SSM_EXPAND,
    )

### 6C. Encoder-decoder Transformer

Retrieval uses the bidirectional **encoder** outputs (`encode()`); inputs must be ≤ `max_input_length` = 1024 tokens.

In [ ]:
import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class EncoderDecoderConfig:
    vocab_size: int
    d_model: int
    d_kv: int
    d_ff: int
    num_layers: int
    num_decoder_layers: int
    num_heads: int
    dropout_rate: float = 0.1
    layer_norm_epsilon: float = 1e-6
    relative_attention_num_buckets: int = 32
    relative_attention_max_distance: int = 128
    feed_forward_proj: str = "relu"
    dense_act_fn: str = "relu"
    tie_word_embeddings: bool = True
    scale_decoder_outputs: bool = True
    initializer_factor: float = 1.0
    pad_token_id: int = 0
    eos_token_id: int = 3
    decoder_start_token_id: int = 0
    max_input_length: int = 1024
    max_target_length: int = 1024
    use_cache: bool = False

    @property
    def inner_dim(self):
        return self.num_heads * self.d_kv

    @property
    def is_gated_act(self):
        return self.feed_forward_proj.startswith("gated-")


class RMSNorm(nn.Module):
    """Scale-only RMS normalization (no mean subtraction, no bias)."""

    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.variance_epsilon = eps

    def forward(self, hidden_states):
        variance = hidden_states.float().pow(2).mean(dim=-1, keepdim=True)
        normalized = hidden_states * torch.rsqrt(
            variance + self.variance_epsilon
        ).to(dtype=hidden_states.dtype)
        return normalized * self.weight.to(dtype=hidden_states.dtype)


class RelativePositionBias(nn.Module):
    """Bucketed relative-position bias: log-spaced buckets, one learned bias per head."""

    def __init__(
        self,
        num_buckets,
        num_heads,
        max_distance,
        bidirectional,
    ):
        super().__init__()
        if num_buckets < 2:
            raise ValueError("relative_attention_num_buckets must be >= 2")
        if max_distance < 2:
            raise ValueError("relative_attention_max_distance must be >= 2")
        self.num_buckets = num_buckets
        self.num_heads = num_heads
        self.max_distance = max_distance
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding(num_buckets, num_heads)

    @staticmethod
    def relative_position_bucket(
        relative_position,
        bidirectional=True,
        num_buckets=32,
        max_distance=128,
    ):
        """Map key_position - query_position to a relative-position bucket."""
        bucket = 0
        if bidirectional:
            num_buckets //= 2
            bucket = bucket + (
                (relative_position > 0).to(torch.long) * num_buckets
            )
            relative_position = torch.abs(relative_position)
        else:
            relative_position = -torch.minimum(
                relative_position, torch.zeros_like(relative_position)
            )

        max_exact = num_buckets // 2
        is_small = relative_position < max_exact
        relative_position_if_large = max_exact + (
            torch.log(
                relative_position.float().clamp(min=1) / max_exact
            )
            / math.log(max_distance / max_exact)
            * (num_buckets - max_exact)
        ).to(torch.long)
        relative_position_if_large = torch.clamp(
            relative_position_if_large, max=num_buckets - 1
        )
        bucket = bucket + torch.where(
            is_small, relative_position, relative_position_if_large
        )
        return bucket

    def forward(self, query_length, key_length, device=None):
        if device is None:
            device = self.embedding.weight.device
        query_positions = torch.arange(
            query_length, dtype=torch.long, device=device
        )[:, None]
        key_positions = torch.arange(
            key_length, dtype=torch.long, device=device
        )[None, :]
        relative_position = key_positions - query_positions
        buckets = self.relative_position_bucket(
            relative_position,
            bidirectional=self.bidirectional,
            num_buckets=self.num_buckets,
            max_distance=self.max_distance,
        )
        values = self.embedding(buckets)
        return values.permute(2, 0, 1).unsqueeze(0)


class MultiHeadAttention(nn.Module):
    """Multi-head attention with bias-free Q/K/V/O projections."""

    def __init__(
        self,
        config: EncoderDecoderConfig,
        causal=False,
        relative_bias_bidirectional=None,
    ):
        super().__init__()
        self.d_model = config.d_model
        self.d_kv = config.d_kv
        self.num_heads = config.num_heads
        self.inner_dim = config.inner_dim
        self.causal = causal

        self.q = nn.Linear(self.d_model, self.inner_dim, bias=False)
        self.k = nn.Linear(self.d_model, self.inner_dim, bias=False)
        self.v = nn.Linear(self.d_model, self.inner_dim, bias=False)
        self.o = nn.Linear(self.inner_dim, self.d_model, bias=False)
        self.dropout = nn.Dropout(config.dropout_rate)
        self.relative_attention_bias = None
        if relative_bias_bidirectional is not None:
            self.relative_attention_bias = RelativePositionBias(
                num_buckets=config.relative_attention_num_buckets,
                num_heads=config.num_heads,
                max_distance=config.relative_attention_max_distance,
                bidirectional=relative_bias_bidirectional,
            )

    def _split_heads(self, projected):
        batch_size, seq_len, _ = projected.shape
        return projected.view(
            batch_size, seq_len, self.num_heads, self.d_kv
        ).transpose(1, 2)

    def forward(
        self,
        hidden_states,
        key_value_states=None,
        attention_mask=None,
        position_bias=None,
    ):
        if key_value_states is None:
            key_value_states = hidden_states
        query = self._split_heads(self.q(hidden_states))
        key = self._split_heads(self.k(key_value_states))
        value = self._split_heads(self.v(key_value_states))

        scores = torch.matmul(query, key.transpose(-2, -1))
        query_length, key_length = scores.shape[-2:]

        if position_bias is None and self.relative_attention_bias is not None:
            position_bias = self.relative_attention_bias(
                query_length, key_length, device=scores.device
            )
        if position_bias is not None:
            scores = scores + position_bias.to(dtype=scores.dtype)

        if self.causal:
            if query_length != key_length:
                raise ValueError(
                    "This implementation does not use a KV cache; "
                    "causal self-attention requires equal query/key lengths."
                )
            causal_allowed = torch.ones(
                query_length,
                key_length,
                dtype=torch.bool,
                device=scores.device,
            ).tril()
            scores = scores.masked_fill(
                ~causal_allowed.view(1, 1, query_length, key_length),
                torch.finfo(scores.dtype).min,
            )

        if attention_mask is not None:
            if attention_mask.ndim != 2 or attention_mask.shape[-1] != key_length:
                raise ValueError(
                    "attention_mask must have shape [batch, key_length]"
                )
            key_is_valid = attention_mask.to(
                device=scores.device, dtype=torch.bool
            ).view(scores.shape[0], 1, 1, key_length)
            scores = scores.masked_fill(
                ~key_is_valid, torch.finfo(scores.dtype).min
            )

        probabilities = F.softmax(scores.float(), dim=-1).to(
            dtype=scores.dtype
        )
        probabilities = self.dropout(probabilities)
        attended = torch.matmul(probabilities, value)
        attended = attended.transpose(1, 2).contiguous().view(
            attended.shape[0], query_length, self.inner_dim
        )
        return self.o(attended)


class EncoderSelfAttention(MultiHeadAttention):
    def __init__(self, config, has_relative_attention_bias=False):
        super().__init__(
            config,
            causal=False,
            relative_bias_bidirectional=(
                True if has_relative_attention_bias else None
            ),
        )


class DecoderSelfAttention(MultiHeadAttention):
    def __init__(self, config, has_relative_attention_bias=False):
        super().__init__(
            config,
            causal=True,
            relative_bias_bidirectional=(
                False if has_relative_attention_bias else None
            ),
        )


class CrossAttention(MultiHeadAttention):
    def __init__(self, config):
        super().__init__(config, causal=False)


class EncDecFeedForward(nn.Module):
    """Feed-forward layer: dense-act-dense, or gated (two input projections)."""

    def __init__(self, config: EncoderDecoderConfig):
        super().__init__()
        self.feed_forward_proj = config.feed_forward_proj
        self.activation_name = config.dense_act_fn
        if config.is_gated_act:
            self.wi_0 = nn.Linear(config.d_model, config.d_ff, bias=False)
            self.wi_1 = nn.Linear(config.d_model, config.d_ff, bias=False)
        else:
            self.wi = nn.Linear(config.d_model, config.d_ff, bias=False)
        self.wo = nn.Linear(config.d_ff, config.d_model, bias=False)
        self.dropout = nn.Dropout(config.dropout_rate)

    def _activation(self, x):
        if self.activation_name == "relu":
            return F.relu(x)
        if self.activation_name in ("gelu", "gelu_new"):
            return F.gelu(x, approximate="tanh" if self.activation_name == "gelu_new" else "none")
        if self.activation_name in ("silu", "swish"):
            return F.silu(x)
        raise ValueError(
            f"Unsupported dense_act_fn: {self.activation_name}"
        )

    def forward(self, hidden_states):
        if self.feed_forward_proj.startswith("gated-"):
            hidden_states = self._activation(self.wi_0(hidden_states)) * self.wi_1(
                hidden_states
            )
        else:
            hidden_states = self._activation(self.wi(hidden_states))
        hidden_states = self.dropout(hidden_states)
        return self.wo(hidden_states)


class EncoderBlock(nn.Module):
    def __init__(self, config: EncoderDecoderConfig, layer_idx: int):
        super().__init__()
        self.self_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.self_attention = EncoderSelfAttention(
            config,
            has_relative_attention_bias=(layer_idx == 0),
        )
        self.feed_forward_norm = RMSNorm(
            config.d_model, config.layer_norm_epsilon
        )
        self.feed_forward = EncDecFeedForward(config)
        self.dropout = nn.Dropout(config.dropout_rate)

    def forward(
        self, hidden_states, attention_mask=None, position_bias=None,
        return_debug=False
    ):
        debug = {}
        attention_output = self.self_attention(
            self.self_norm(hidden_states),
            attention_mask=attention_mask,
            position_bias=position_bias,
        )
        hidden_states = hidden_states + self.dropout(attention_output)
        if return_debug:
            debug["self_attention_output"] = hidden_states

        feed_forward_output = self.feed_forward(
            self.feed_forward_norm(hidden_states)
        )
        hidden_states = hidden_states + self.dropout(feed_forward_output)
        if return_debug:
            debug["feed_forward_output"] = hidden_states
            return hidden_states, debug
        return hidden_states


class DecoderBlock(nn.Module):
    def __init__(self, config: EncoderDecoderConfig, layer_idx: int):
        super().__init__()
        self.self_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.self_attention = DecoderSelfAttention(
            config,
            has_relative_attention_bias=(layer_idx == 0),
        )
        self.cross_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.cross_attention = CrossAttention(config)
        self.feed_forward_norm = RMSNorm(
            config.d_model, config.layer_norm_epsilon
        )
        self.feed_forward = EncDecFeedForward(config)
        self.dropout = nn.Dropout(config.dropout_rate)

    def forward(
        self,
        hidden_states,
        encoder_hidden_states,
        encoder_attention_mask=None,
        decoder_attention_mask=None,
        position_bias=None,
        return_debug=False,
    ):
        debug = {}
        self_output = self.self_attention(
            self.self_norm(hidden_states),
            attention_mask=decoder_attention_mask,
            position_bias=position_bias,
        )
        hidden_states = hidden_states + self.dropout(self_output)
        if return_debug:
            debug["self_attention_output"] = hidden_states

        cross_output = self.cross_attention(
            self.cross_norm(hidden_states),
            key_value_states=encoder_hidden_states,
            attention_mask=encoder_attention_mask,
        )
        hidden_states = hidden_states + self.dropout(cross_output)
        if return_debug:
            debug["cross_attention_output"] = hidden_states

        feed_forward_output = self.feed_forward(
            self.feed_forward_norm(hidden_states)
        )
        hidden_states = hidden_states + self.dropout(feed_forward_output)
        if return_debug:
            debug["feed_forward_output"] = hidden_states
            return hidden_states, debug
        return hidden_states


class Encoder(nn.Module):
    def __init__(self, config: EncoderDecoderConfig):
        super().__init__()
        self.config = config
        self.blocks = nn.ModuleList(
            EncoderBlock(config, layer_idx)
            for layer_idx in range(config.num_layers)
        )
        self.final_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.dropout = nn.Dropout(config.dropout_rate)

    def _run(self, inputs_embeds, attention_mask=None, debug=False):
        hidden_states = self.dropout(inputs_embeds)
        position_bias = self.blocks[0].self_attention.relative_attention_bias(
            hidden_states.shape[1],
            hidden_states.shape[1],
            device=hidden_states.device,
        )
        states = {"embeddings": hidden_states}
        for layer_idx, block in enumerate(self.blocks):
            if debug:
                hidden_states, block_debug = block(
                    hidden_states,
                    attention_mask=attention_mask,
                    position_bias=position_bias,
                    return_debug=True,
                )
                states[f"block_{layer_idx}_self_attention"] = (
                    block_debug["self_attention_output"]
                )
                states[f"block_{layer_idx}_feed_forward"] = (
                    block_debug["feed_forward_output"]
                )
            else:
                hidden_states = block(
                    hidden_states,
                    attention_mask=attention_mask,
                    position_bias=position_bias,
                )
        hidden_states = self.dropout(self.final_norm(hidden_states))
        states["final_norm"] = hidden_states
        return (hidden_states, states) if debug else hidden_states

    def forward(self, inputs_embeds, attention_mask=None):
        return self._run(inputs_embeds, attention_mask, debug=False)

    def debug_hidden_states(self, inputs_embeds, attention_mask=None):
        return self._run(inputs_embeds, attention_mask, debug=True)


class Decoder(nn.Module):
    def __init__(self, config: EncoderDecoderConfig):
        super().__init__()
        self.config = config
        self.blocks = nn.ModuleList(
            DecoderBlock(config, layer_idx)
            for layer_idx in range(config.num_decoder_layers)
        )
        self.final_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.dropout = nn.Dropout(config.dropout_rate)

    def _run(
        self,
        inputs_embeds,
        encoder_hidden_states,
        encoder_attention_mask=None,
        decoder_attention_mask=None,
        debug=False,
    ):
        hidden_states = self.dropout(inputs_embeds)
        position_bias = self.blocks[0].self_attention.relative_attention_bias(
            hidden_states.shape[1],
            hidden_states.shape[1],
            device=hidden_states.device,
        )
        states = {"embeddings": hidden_states}
        for layer_idx, block in enumerate(self.blocks):
            if debug:
                hidden_states, block_debug = block(
                    hidden_states,
                    encoder_hidden_states,
                    encoder_attention_mask=encoder_attention_mask,
                    decoder_attention_mask=decoder_attention_mask,
                    position_bias=position_bias,
                    return_debug=True,
                )
                for name, value in block_debug.items():
                    states[f"block_{layer_idx}_{name}"] = value
            else:
                hidden_states = block(
                    hidden_states,
                    encoder_hidden_states,
                    encoder_attention_mask=encoder_attention_mask,
                    decoder_attention_mask=decoder_attention_mask,
                    position_bias=position_bias,
                )
        hidden_states = self.dropout(self.final_norm(hidden_states))
        states["final_norm"] = hidden_states
        return (hidden_states, states) if debug else hidden_states

    def forward(
        self,
        inputs_embeds,
        encoder_hidden_states,
        encoder_attention_mask=None,
        decoder_attention_mask=None,
    ):
        return self._run(
            inputs_embeds,
            encoder_hidden_states,
            encoder_attention_mask,
            decoder_attention_mask,
            debug=False,
        )

    def debug_hidden_states(
        self,
        inputs_embeds,
        encoder_hidden_states,
        encoder_attention_mask=None,
        decoder_attention_mask=None,
    ):
        return self._run(
            inputs_embeds,
            encoder_hidden_states,
            encoder_attention_mask,
            decoder_attention_mask,
            debug=True,
        )


class EncoderDecoderTransformer(nn.Module):
    """Encoder-decoder Transformer, implemented with raw PyTorch layers."""

    def __init__(self, config: EncoderDecoderConfig):
        super().__init__()
        if config.d_model <= 0 or config.d_kv <= 0:
            raise ValueError("d_model and d_kv must be positive")
        if config.num_heads <= 0 or config.num_layers <= 0:
            raise ValueError("num_heads and num_layers must be positive")
        if config.num_decoder_layers <= 0:
            raise ValueError("num_decoder_layers must be positive")
        self.config = config
        self.shared_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.encoder = Encoder(config)
        self.decoder = Decoder(config)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self._initialize_weights()
        if config.tie_word_embeddings:
            # One embedding matrix is shared by the encoder, decoder and LM head.
            self.lm_head.weight = self.shared_embedding.weight

    def _initialize_weights(self):
        factor = self.config.initializer_factor
        with torch.no_grad():
            self.shared_embedding.weight.normal_(mean=0.0, std=factor)
            for module in self.modules():
                if isinstance(module, RMSNorm):
                    module.weight.fill_(1.0)
                elif isinstance(module, RelativePositionBias):
                    module.embedding.weight.normal_(mean=0.0, std=factor)
                elif isinstance(module, MultiHeadAttention):
                    module.q.weight.normal_(
                        mean=0.0,
                        std=factor
                        * (self.config.d_model * self.config.d_kv) ** -0.5,
                    )
                    for projection in (module.k, module.v):
                        projection.weight.normal_(
                            mean=0.0,
                            std=factor * self.config.d_model ** -0.5,
                        )
                    module.o.weight.normal_(
                        mean=0.0,
                        std=factor * self.config.inner_dim ** -0.5,
                    )
                elif isinstance(module, EncDecFeedForward):
                    if self.config.is_gated_act:
                        module.wi_0.weight.normal_(
                            mean=0.0,
                            std=factor * self.config.d_model ** -0.5,
                        )
                        module.wi_1.weight.normal_(
                            mean=0.0,
                            std=factor * self.config.d_model ** -0.5,
                        )
                    else:
                        module.wi.weight.normal_(
                            mean=0.0,
                            std=factor * self.config.d_model ** -0.5,
                        )
                    module.wo.weight.normal_(
                        mean=0.0,
                        std=factor * self.config.d_ff ** -0.5,
                    )

    def _validate_input_ids(self, input_ids, max_length, name):
        if input_ids.ndim != 2:
            raise ValueError(f"{name} must have shape [batch, sequence]")
        if input_ids.shape[1] == 0:
            raise ValueError(f"{name} must not be empty")
        if input_ids.shape[1] > max_length:
            raise ValueError(
                f"{name} length {input_ids.shape[1]} exceeds {max_length}"
            )

    def encode(self, input_ids, attention_mask=None):
        """Return final encoder states with shape [B, L, d_model]."""
        self._validate_input_ids(
            input_ids, self.config.max_input_length, "input_ids"
        )
        if attention_mask is not None and attention_mask.shape != input_ids.shape:
            raise ValueError("attention_mask must match input_ids shape")
        return self.encoder(
            self.shared_embedding(input_ids),
            attention_mask=attention_mask,
        )

    def decode(
        self,
        decoder_input_ids,
        encoder_hidden_states,
        attention_mask=None,
        decoder_attention_mask=None,
    ):
        self._validate_input_ids(
            decoder_input_ids,
            self.config.max_target_length,
            "decoder_input_ids",
        )
        if decoder_input_ids.shape[0] != encoder_hidden_states.shape[0]:
            raise ValueError(
                "decoder and encoder batch dimensions must match"
            )
        return self.decoder(
            self.shared_embedding(decoder_input_ids),
            encoder_hidden_states,
            encoder_attention_mask=attention_mask,
            decoder_attention_mask=decoder_attention_mask,
        )

    def shift_right(self, labels):
        if labels.ndim != 2:
            raise ValueError("labels must have shape [batch, target_length]")
        if labels.shape[1] == 0:
            raise ValueError("labels must not be empty")
        shifted = labels.new_full(labels.shape, self.config.pad_token_id)
        shifted[:, 0] = self.config.decoder_start_token_id
        shifted[:, 1:] = labels[:, :-1]
        shifted.masked_fill_(shifted == -100, self.config.pad_token_id)
        return shifted

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        decoder_input_ids=None,
        decoder_attention_mask=None,
        encoder_hidden_states=None,
    ):
        if encoder_hidden_states is None:
            if input_ids is None:
                raise ValueError(
                    "input_ids are required when encoder states are not supplied"
                )
            encoder_hidden_states = self.encode(input_ids, attention_mask)
        elif input_ids is not None:
            self._validate_input_ids(
                input_ids, self.config.max_input_length, "input_ids"
            )

        if decoder_input_ids is None:
            if labels is None:
                raise ValueError(
                    "Pass decoder_input_ids or labels to run the decoder"
                )
            decoder_input_ids = self.shift_right(labels)
        if decoder_input_ids.shape[1] > self.config.max_target_length:
            raise ValueError("decoder_input_ids exceed max_target_length")

        decoder_hidden_states = self.decode(
            decoder_input_ids,
            encoder_hidden_states,
            attention_mask=attention_mask,
            decoder_attention_mask=decoder_attention_mask,
        )
        if (
            self.config.tie_word_embeddings
            and self.config.scale_decoder_outputs
        ):
            decoder_hidden_states = decoder_hidden_states * (
                self.config.d_model ** -0.5
            )
        logits = self.lm_head(decoder_hidden_states)
        loss = None
        if labels is not None:
            if labels.shape != logits.shape[:2]:
                raise ValueError(
                    "labels must match the decoder logits sequence dimensions"
                )
            loss_labels = labels.masked_fill(
                labels == self.config.pad_token_id, -100
            )
            loss = F.cross_entropy(
                logits.float().reshape(-1, self.config.vocab_size),
                loss_labels.reshape(-1),
                ignore_index=-100,
            )
        return {
            "loss": loss,
            "logits": logits,
            "encoder_hidden_states": encoder_hidden_states,
        }



# Phase-3 uses the encoder representation from this pretrained Phase-1 model.
def _encoder_decoder_encode(self, input_ids, attention_mask=None):
    return self.encode(input_ids, attention_mask)

# The original Phase-1 class already defines encode(); this alias documents the
# task-facing API without changing its state-dict parameter names.

ENCDEC_CONFIG = EncoderDecoderConfig(
    vocab_size=16000,
    d_model=512,
    d_kv=64,
    d_ff=2048,
    num_layers=6,
    num_decoder_layers=6,
    num_heads=8,
    dropout_rate=0.1,
    layer_norm_epsilon=1e-6,
    relative_attention_num_buckets=32,
    relative_attention_max_distance=128,
    feed_forward_proj="relu",
    dense_act_fn="relu",
    tie_word_embeddings=True,
    scale_decoder_outputs=True,
    initializer_factor=1.0,
    pad_token_id=PAD_ID,
    eos_token_id=EOS_ID,
    decoder_start_token_id=PAD_ID,
    max_input_length=1024,
    max_target_length=1024,
)


def build_encdec_backbone():
    return EncoderDecoderTransformer(ENCDEC_CONFIG)

### 6D. CNN + Causal Self-Attention

Its attention class is named `CNNCausalSelfAttention` so it does not overwrite the decoder's `CausalSelfAttention` (parameter names unchanged).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
#1
class CausalConvBlock(nn.Module):


        def __init__(self, dim, kernel_size):
            super().__init__()
        
            self.kernel_size = kernel_size
        
            self.conv = nn.Conv1d(
                in_channels=dim,
                out_channels=dim,
                kernel_size=kernel_size
            )
        
        def forward(self, x):
        
            # x: [B, T, C]
        
            # Step 1: transpose
            x = x.transpose(1, 2)  # [B, C, T]
        
            # Step 2: left padding only
            x = F.pad(x, (self.kernel_size - 1, 0))
        
            # Step 3: convolution
            x = self.conv(x)
        
            # Step 4: transpose back
            x = x.transpose(1, 2)  # [B, T, C]
        
            return x


import math
class CNNCausalSelfAttention(nn.Module):


    def __init__(self, dim, n_heads):
        super().__init__()
    
        assert dim % n_heads == 0
    
        self.dim = dim
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
    
        # Step 1: Linear projections for Q, K, V
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
    
        # Step 2: Final projection
        self.out_proj = nn.Linear(dim, dim)
    
    
    def forward(self, x):
    
        # x: [B, T, C]
        B, T, C = x.shape
    
        # Step 3: Compute Q, K, V
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
    
        # Step 4: Split into heads
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    
        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=0.0,
            is_causal=True
        )
    
        # Step 10: Merge heads
        out = out.transpose(1, 2).contiguous().view(B, T, C)
    
        # Step 11: Final projection
        out = self.out_proj(out)
    
        return out


class HybridBlock(nn.Module):

        
        def __init__(self, dim, n_heads, kernel_size, ff_mult=4):
            super().__init__()
        
            # Step 1: LayerNorm before CNN
            self.norm1 = nn.LayerNorm(dim)
        
            # Step 2: Causal CNN block
            self.conv = CausalConvBlock(dim, kernel_size)
        
            # Step 3: LayerNorm before Attention
            self.norm2 = nn.LayerNorm(dim)
        
            # Step 4: Causal Self-Attention
            self.attn = CNNCausalSelfAttention(dim, n_heads)
        
            # Step 5: LayerNorm before FFN
            self.norm3 = nn.LayerNorm(dim)
        
            # Step 6: Feed Forward Network
            self.ffn = nn.Sequential(
                nn.Linear(dim, ff_mult * dim),
                nn.GELU(),
                nn.Linear(ff_mult * dim, dim)
            )
        
        
        def forward(self, x):
        
            # x shape: [B, T, C]
        
            # Step 7: CNN + residual
            x = x + self.conv(self.norm1(x))
        
            # Step 8: Attention + residual
            x = x + self.attn(self.norm2(x))
        
            # Step 9: FFN + residual
            x = x + self.ffn(self.norm3(x))
        
            return x



class CNNAttentionLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        dim,
        n_layers,
        n_heads,
        max_seq_len,
        kernel_size=3,
        ff_mult=4
    ):
        super().__init__()

        # Save important model settings
        self.vocab_size = vocab_size
        self.dim = dim
        self.max_seq_len = max_seq_len

        # Converts each token ID into a dim-dimensional vector
        # Input shape:  [B, T]
        # Output shape: [B, T, dim]
        self.token_emb = nn.Embedding(vocab_size, dim)

        # Learnable positional embeddings
        # Gives the model information about token position in sequence
        self.pos_emb = nn.Embedding(max_seq_len, dim)

        # Stack multiple CNN + Attention hybrid blocks
        self.blocks = nn.ModuleList([
            HybridBlock(
                dim,
                n_heads,
                kernel_size,
                ff_mult
            )
            for _ in range(n_layers)
        ])

        # Final normalization before vocabulary prediction
        self.final_norm = nn.LayerNorm(dim)

        # Language modeling head
        # Converts hidden representation into logits over vocabulary
        #
        # [B, T, dim] -> [B, T, vocab_size]
        self.lm_head = nn.Linear(dim, vocab_size)


    def forward(self, input_ids, labels=None, **kwargs):

            B, T = input_ids.shape
    
            assert T <= self.max_seq_len
    
            # Token embeddings: [B,T] -> [B,T,C]
            tok = self.token_emb(input_ids)
    
            # Position IDs: [0,1,...,T-1]
            positions = torch.arange(
                T,
                device=input_ids.device
            )
    
            # Position embeddings: [T] -> [T,C]
            pos = self.pos_emb(positions)
    
            # Broadcasting gives [B,T,C]
            x = tok + pos
    
            # CNN + Attention hybrid blocks
            for block in self.blocks:
                x = block(x)
    
            # Final normalization
            x = self.final_norm(x)
    
            # Vocabulary predictions
            # [B,T,C] -> [B,T,V]
            logits = self.lm_head(x)
    
            loss = None
            
            if labels is not None:
            
                # Create targets with the same sequence length
                targets = torch.full_like(
                    labels,
                    -100
                )
            
                # Position t predicts token t+1
                targets[:, :-1] = labels[:, 1:]
            
                # The final position has target -100,
                # so cross entropy ignores it
                loss = F.cross_entropy(
                    logits.reshape(-1, self.vocab_size),
                    targets.reshape(-1),
                    ignore_index=-100
                )
            
                # Trainer only needs loss during training
                return {
                    "loss": loss
                }
            
            # During generation we need logits
            return {
                "logits": logits
            }

# Match the common Phase-1 backbone interface used by Phase 3. The model is
# causal, so right-padding tokens cannot affect earlier sentence token states.
def _cnn_encode(self, input_ids, attention_mask=None):
    batch_size, seq_len = input_ids.shape
    if seq_len > self.max_seq_len:
        raise ValueError(
            f"Sequence length {seq_len} exceeds max_seq_len={self.max_seq_len}"
        )
    token_states = self.token_emb(input_ids)
    position_ids = torch.arange(seq_len, device=input_ids.device)
    hidden_states = token_states + self.pos_emb(position_ids).unsqueeze(0)
    for block in self.blocks:
        hidden_states = block(hidden_states)
    return self.final_norm(hidden_states)

CNNAttentionLM.encode = _cnn_encode
CNN_DIM = 640
CNN_N_LAYERS = 8
CNN_N_HEADS = 8
CNN_MAX_SEQ_LEN = 2048
CNN_KERNEL_SIZE = 3
CNN_FF_MULT = 4

def build_cnn_backbone():
    return CNNAttentionLM(
        vocab_size=16000,
        dim=CNN_DIM,
        n_layers=CNN_N_LAYERS,
        n_heads=CNN_N_HEADS,
        max_seq_len=CNN_MAX_SEQ_LEN,
        kernel_size=CNN_KERNEL_SIZE,
        ff_mult=CNN_FF_MULT,
    )

In [ ]:
# Maximum sequence length accepted by each backbone (chunk / prompt sizes must respect it)
BACKBONE_MAX_TOKENS = {
    "decoder": decoder_config.context_length,       # 2048
    "ssm": SSM_MAX_SEQ_LEN,                          # 2048
    "encdec": ENCDEC_CONFIG.max_input_length,        # 1024 (encoder input)
    "cnn": CNN_MAX_SEQ_LEN,                          # 2048
}
print(BACKBONE_MAX_TOKENS)

## 7. Backbone registry and checkpoint checks

`adapter_targets` are the `nn.Linear` module-name suffixes wrapped by IA³ / LORMA; `ia3_feedforward` must be a subset of them
(for the CNN the feed-forward layers `ffn.0` / `ffn.2` are targeted together with the attention projections).
`.safetensors`, `.bin`/`.pt` files and checkpoint directories are accepted.

In [ ]:
BACKBONES = {}


def register_backbone(
    name,
    builder,
    hidden_dim,
    checkpoint_path=None,
    adapter_targets=None,
    ia3_feedforward=None,
):
    BACKBONES[name] = {
        "builder": builder,
        "hidden_dim": hidden_dim,
        "checkpoint_path": checkpoint_path,
        # nn.Linear module-name suffixes for LoRA / rsLoRA / IA3 / LORMA
        "adapter_targets": adapter_targets or [],
        # subset of adapter_targets treated as feed-forward by IA3
        "ia3_feedforward": ia3_feedforward or [],
    }


def _load_state(path):
    from pathlib import Path

    path = Path(path)
    if path.is_dir():
        safe_path = path / "model.safetensors"
        bin_path = path / "pytorch_model.bin"
        if safe_path.is_file():
            path = safe_path
        elif bin_path.is_file():
            path = bin_path
        else:
            raise FileNotFoundError(
                f"No model.safetensors or pytorch_model.bin in {path}"
            )

    if str(path).endswith(".safetensors"):
        from safetensors.torch import load_file
        state = load_file(str(path), device="cpu")
    else:
        try:
            state = torch.load(path, map_location="cpu", weights_only=True)
        except TypeError:
            state = torch.load(path, map_location="cpu")

    if isinstance(state, dict):
        for wrapper_key in ("model_state_dict", "state_dict"):
            if wrapper_key in state and isinstance(state[wrapper_key], dict):
                state = state[wrapper_key]
                break
    return state

def load_backbone(name, checkpoint_path=None, device="cpu", strict=True):
    if name not in BACKBONES:
        raise ValueError(f"Unknown backbone: {name}. Available: {list(BACKBONES)}")

    info = BACKBONES[name]
    model = info["builder"]()

    ckpt = checkpoint_path if checkpoint_path is not None else info["checkpoint_path"]
    if ckpt is None:
        raise ValueError(f"No checkpoint supplied for backbone '{name}'")

    result = model.load_state_dict(_load_state(ckpt), strict=strict)

    print("Backbone:", name, "| checkpoint:", ckpt)
    print("Missing keys   :", result.missing_keys)
    print("Unexpected keys:", result.unexpected_keys)

    return model.to(device)

In [ ]:
register_backbone(
    name="decoder",
    builder=lambda: DecoderOnlyTransformer(decoder_config),
    hidden_dim=decoder_config.n_embd,
    checkpoint_path=DECODER_CHECKPOINT,
    adapter_targets=["qkv", "out_proj", "fc_in", "fc_out"],
    ia3_feedforward=["fc_out"],
)

register_backbone(
    name="ssm",
    builder=build_ssm_backbone,
    hidden_dim=SSM_DIM,
    checkpoint_path=SSM_CHECKPOINT,
    adapter_targets=["in_proj", "out_proj"],
    ia3_feedforward=["out_proj"],
)

register_backbone(
    name="encdec",
    builder=build_encdec_backbone,
    hidden_dim=ENCDEC_CONFIG.d_model,
    checkpoint_path=ENCDEC_CHECKPOINT,
    # wi_0 / wi_1 exist only for gated feed-forward configs; unmatched suffixes are ignored
    adapter_targets=["q", "k", "v", "o", "wi", "wi_0", "wi_1", "wo"],
    ia3_feedforward=["wi", "wi_0", "wi_1", "wo"],
)

register_backbone(
    name="cnn",
    builder=build_cnn_backbone,
    hidden_dim=CNN_DIM,
    checkpoint_path=CNN_CHECKPOINT,
    adapter_targets=["q_proj", "k_proj", "v_proj", "out_proj", "ffn.0", "ffn.2"],
    ia3_feedforward=["ffn.2"],
)

assert set(BACKBONES) == {"decoder", "ssm", "encdec", "cnn"}, list(BACKBONES)
for _name, _info in BACKBONES.items():
    assert set(_info["ia3_feedforward"]) <= set(_info["adapter_targets"]), _name
print("Registered backbones:", list(BACKBONES))

In [ ]:
# Checkpoint presence (strict loading + encode() shapes are tested in section 13).
for name, info in BACKBONES.items():
    status = "found" if Path(info["checkpoint_path"]).exists() else "MISSING"
    print(f"{name:10s} {status:8s} {info['checkpoint_path']}")

## 8. Phase-7 model wrapper

Bi-encoder: `chunks → backbone.encode() → chunk vectors → document vector (pooling) → projection → L2-normalise`;
`scores = q · cᵀ / τ` (cosine / temperature); loss = multi-positive InfoNCE (contrastive) over the batch's candidates.
The training loss adds `0.3 ×` a pairwise ranking term: every positive should beat the 3 hardest in-batch negatives by a
cosine margin of 0.1 (softplus).

Token pooling (`pool_tokens`) is defined first because the wrapper uses it.

In [ ]:
POOLING_STRATEGIES = ("last_token", "mean", "attention")
DEFAULT_POOLING = {"decoder": "last_token", "ssm": "last_token", "cnn": "last_token", "encdec": "mean"}


def pool_tokens(h, mask, strategy, pooler=None):
    """
    h       : [N, L, D] token states,   mask : [N, L] (1 = real token; every row has >= 1 real token)
    strategy: "last_token" | "mean" | "attention" (attention needs pooler = nn.Linear(D, 1, bias=False))
    returns : [N, D]
    """
    if strategy == "last_token":
        last_idx = mask.long().sum(dim=1) - 1
        rows = torch.arange(h.size(0), device=h.device)
        return h[rows, last_idx]
    if strategy == "mean":
        mask_f = mask.unsqueeze(-1).to(h.dtype)
        return (h * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1)
    if strategy == "attention":
        assert pooler is not None, "attention pooling needs a learned pooler"
        scores = pooler(h).squeeze(-1)
        scores = scores.masked_fill(mask == 0, torch.finfo(scores.dtype).min)
        weights = torch.softmax(scores, dim=-1)
        return (h * weights.unsqueeze(-1)).sum(dim=1)
    raise ValueError(f"Unknown pooling strategy: {strategy}")

In [ ]:
class Phase7PCRModel(nn.Module):
    """
    Siamese bi-encoder for Phase 7 legal document retrieval.

    encode_documents:
        [N, C, L] -> [N, E] normalized embeddings

    forward:
        returns {"loss": scalar or None, "scores": [B, M]}
    """

    def __init__(
        self,
        backbone,
        hidden_dim,
        embed_dim=256,
        temperature=0.05,
        dropout=0.1,
        pooling_strategy="last_token",
        doc_pooling="mean",
    ):
        super().__init__()

        self.backbone = backbone
        self.hidden_dim = hidden_dim
        self.embed_dim = embed_dim
        self.temperature = temperature

        assert pooling_strategy in POOLING_STRATEGIES
        assert doc_pooling in ("mean", "max", "attention")
        assert temperature > 0

        self.pooling_strategy = pooling_strategy
        self.doc_pooling = doc_pooling

        # Learned token-level attention pooling, if required.
        if pooling_strategy == "attention":
            self.chunk_pool = nn.Linear(
                hidden_dim, 1, bias=False
            )
        else:
            self.chunk_pool = nn.Identity()

        # Learned document-level attention pooling, if required.
        if doc_pooling == "attention":
            self.doc_pool = nn.Linear(
                hidden_dim, 1, bias=False
            )
        else:
            self.doc_pool = nn.Identity()

        # Projection head: hidden_dim -> embed_dim.
        self.proj = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
        )

        # Backbone sub-batch size to limit memory consumption.
        self.encode_batch_size = 2

    def head_module_names(self):
        return ["chunk_pool", "doc_pool", "proj"]

    def encode_documents(
        self,
        input_ids,
        attention_mask,
        chunk_mask,
    ):
        """
        input_ids      : [N, C, L]
        attention_mask : [N, C, L]
        chunk_mask     : [N, C]

        returns        : [N, E]
        """

        N, C, L = input_ids.shape

        # Flatten document/chunk dimensions.
        flat_ids = input_ids.reshape(N * C, L)
        flat_attention = attention_mask.reshape(N * C, L)

        # Only encode real chunks.
        valid_chunks = chunk_mask.reshape(-1).bool()

        valid_indices = valid_chunks.nonzero(
            as_tuple=True
        )[0]

        if valid_indices.numel() == 0:
            raise ValueError(
                "encode_documents received no valid chunks."
            )

        # --------------------------------------------------
        # 1. Backbone encoding in memory-safe sub-batches
        # --------------------------------------------------
        chunk_vectors = []

        for start in range(
            0,
            len(valid_indices),
            self.encode_batch_size,
        ):
            indices = valid_indices[
                start:start + self.encode_batch_size
            ]

            ids_batch = flat_ids.index_select(
                0, indices
            )

            attention_batch = flat_attention.index_select(
                0, indices
            )

            hidden_states = self.backbone.encode(
                ids_batch,
                attention_mask=attention_batch,
            )

            # [n_chunks, L, D] -> [n_chunks, D]
            pooled = pool_tokens(
                hidden_states,
                attention_batch,
                self.pooling_strategy,
                pooler=(
                    self.chunk_pool
                    if self.pooling_strategy == "attention"
                    else None
                ),
            )

            chunk_vectors.append(pooled)

        chunk_vectors = torch.cat(
            chunk_vectors,
            dim=0,
        )

        # --------------------------------------------------
        # 2. Restore [N, C, D], filling padded chunks
        # --------------------------------------------------
        all_chunk_vectors = chunk_vectors.new_zeros(
            N * C,
            self.hidden_dim,
        )

        all_chunk_vectors = all_chunk_vectors.index_copy(
            0,
            valid_indices,
            chunk_vectors,
        )

        all_chunk_vectors = all_chunk_vectors.reshape(
            N,
            C,
            self.hidden_dim,
        )

        # --------------------------------------------------
        # 3. Document-level pooling over real chunks
        # --------------------------------------------------
        mask = chunk_mask.bool()

        if self.doc_pooling == "mean":

            mask_f = mask.unsqueeze(-1).to(
                all_chunk_vectors.dtype
            )

            document_vectors = (
                all_chunk_vectors * mask_f
            ).sum(dim=1)

            document_vectors = (
                document_vectors
                / mask_f.sum(dim=1).clamp(min=1)
            )

        elif self.doc_pooling == "max":

            masked_vectors = all_chunk_vectors.masked_fill(
                ~mask.unsqueeze(-1),
                torch.finfo(
                    all_chunk_vectors.dtype
                ).min,
            )

            document_vectors = masked_vectors.max(
                dim=1
            ).values

        elif self.doc_pooling == "attention":

            scores = self.doc_pool(
                all_chunk_vectors
            ).squeeze(-1)

            scores = scores.masked_fill(
                ~mask,
                torch.finfo(scores.dtype).min,
            )

            weights = torch.softmax(
                scores,
                dim=1,
            )

            document_vectors = (
                all_chunk_vectors
                * weights.unsqueeze(-1)
            ).sum(dim=1)

        else:
            raise ValueError(
                f"Unknown doc_pooling: {self.doc_pooling}"
            )

        # --------------------------------------------------
        # 4. Projection and L2 normalization
        # --------------------------------------------------
        embeddings = self.proj(
            document_vectors
        )

        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=-1,
        )

        return embeddings

    def forward(
        self,
        query_input_ids,
        query_attention_mask,
        query_chunk_mask,
        cand_input_ids,
        cand_attention_mask,
        cand_chunk_mask,
        relevance=None,
        **unused,
    ):

        # Query embeddings: [B, E]
        q_embeddings = self.encode_documents(
            query_input_ids,
            query_attention_mask,
            query_chunk_mask,
        )

        # Candidate embeddings: [M, E]
        c_embeddings = self.encode_documents(
            cand_input_ids,
            cand_attention_mask,
            cand_chunk_mask,
        )

        # Similarity scores: [B, M]
        scores = (
            q_embeddings @ c_embeddings.T
        ) / self.temperature

        loss = None

        if relevance is not None:
        
            relevance = relevance.bool()
        
            # Keep only queries having >= 1 positive.
            valid_rows = relevance.any(dim=1)
        
            if valid_rows.any():
        
                valid_scores = scores[valid_rows]
                valid_relevance = relevance[valid_rows]
        
                # ---------------------------------------------
                # 1. Original multi-positive InfoNCE loss
                # ---------------------------------------------
                valid_scores = valid_scores.float()
        
                positive_scores = valid_scores.masked_fill(
                    ~valid_relevance,
                    float("-inf"),
                )
        
                log_positive_sum = torch.logsumexp(
                    positive_scores,
                    dim=1,
                )
        
                log_all_sum = torch.logsumexp(
                    valid_scores,
                    dim=1,
                )
        
                contrastive_loss = (
                    log_all_sum - log_positive_sum
                ).mean()
        
                # ---------------------------------------------
                # 2. Pairwise hard-negative ranking loss
                # ---------------------------------------------
                # Convert temperature-scaled scores back to cosine
                # similarity. Pairwise margin is in cosine units.
                cosine_scores = valid_scores * self.temperature
        
                pairwise_losses = []
        
                num_hard_negatives = 3
                margin = 0.1
        
                for b in range(cosine_scores.size(0)):
        
                    pos_mask = valid_relevance[b]
                    neg_mask = ~pos_mask
        
                    pos_scores = cosine_scores[b][pos_mask]
                    neg_scores = cosine_scores[b][neg_mask]
        
                    if (
                        pos_scores.numel() == 0
                        or neg_scores.numel() == 0
                    ):
                        continue
        
                    # Select the hardest irrelevant candidates.
                    k = min(
                        num_hard_negatives,
                        neg_scores.numel(),
                    )
        
                    hard_neg_scores = torch.topk(
                        neg_scores,
                        k=k,
                        largest=True,
                    ).values
        
                    # Every positive should outrank hard negatives.
                    pairwise_diff = (
                        pos_scores.unsqueeze(1)
                        - hard_neg_scores.unsqueeze(0)
                    )
        
                    ranking_loss = F.softplus(
                        margin - pairwise_diff
                    ).mean()
        
                    pairwise_losses.append(ranking_loss)
        
                if pairwise_losses:
                    ranking_loss = torch.stack(
                        pairwise_losses
                    ).mean()
                else:
                    ranking_loss = contrastive_loss * 0.0
        
                # ---------------------------------------------
                # 3. Combined objective
                # ---------------------------------------------
                loss = (
                    contrastive_loss
                    + 0.3 * ranking_loss
                )
        
            else:
                loss = scores.sum() * 0.0
        return {
            "loss": loss,
            "scores": scores,
        }


def build_phase7_model(
    backbone_name,
    device,
    checkpoint_path=None,
    backbone=None,
    **model_kwargs,
):
    """
    Generic builder: Phase-1 backbone + Phase-7 retrieval head.
    """

    if backbone is None:
        backbone = load_backbone(
            backbone_name,
            checkpoint_path=checkpoint_path,
            device="cpu",
        )

    model = Phase7PCRModel(
        backbone=backbone,
        hidden_dim=BACKBONES[backbone_name]["hidden_dim"],
        **model_kwargs,
    )

    return model.to(device)

## 9. Pooling

For causal backbones (`decoder`, `ssm`, `cnn`) **last-token pooling** is preferred — the final real token has seen the whole chunk.
For the bidirectional `encdec` encoder, **masked mean pooling** is natural. **Attention pooling** adds a learned scorer.
`pool_tokens()` reproduces the three strategies of `Phase3RRModel` so the Phase-7 head can reuse them for token → chunk pooling
(and, with a chunk-level mask, for chunk → document pooling).

Document pooling over real chunks: `doc_pooling` = `mean` (default) / `max` / `attention`.

## 10. Adaptation methods

Four adaptation methods are registered here (`full`, `partial`, `ia3`, `lorma`); knowledge distillation is in section 15.

| key | what is trained | update applied |
|-----|-----------------|----------------|
| `full` | every parameter | — |
| `partial` | last 25 % (≥ 1) of the layers in every `nn.ModuleList` of the backbone + head | — |
| `ia3` | one vector `l` per targeted layer + head | attention-style `y = l ⊙ (W₀x)`; feed-forward `y = W₀(l ⊙ x)` |
| `lorma` | `A ∈ ℝ^(r×d_out)`, `B ∈ ℝ^(d_out×r)` per targeted `nn.Linear` (`B = 0` at init ⇒ identity) + head | `(I + (α/r)·BA)·W₀x + b` (**multiplicative**) |

The retrieval head (`chunk_pool`, `doc_pool`, `proj`) is always trainable. For IA³ the backbone is wrapped in
`PeftBackboneShim` + `get_peft_model`; `.encode()` is forwarded to the Phase-1 model.

> `encdec` note: retrieval uses only the encoder, but the target names also match decoder layers; those adapters receive no gradient.

In [ ]:
class LowRankMulLinear(nn.Module):
    """LORMA: frozen base nn.Linear + trainable low-rank update (from Phase 2)."""

    def __init__(self, base_layer, rank=16, alpha=32.0):
        super().__init__()

        if not isinstance(base_layer, nn.Linear):
            raise TypeError("base_layer must be nn.Linear")

        self.base = base_layer
        for p in self.base.parameters():
            p.requires_grad = False

        self.rank = rank
        self.alpha = alpha
        self.scale = alpha / rank

        in_dim = base_layer.in_features
        out_dim = base_layer.out_features
        device = base_layer.weight.device
        dtype = base_layer.weight.dtype
        #LORA
        # self.A = nn.Parameter(torch.empty(rank, in_dim, device=device, dtype=dtype))
        # self.B = nn.Parameter(torch.zeros(out_dim, rank, device=device, dtype=dtype))
        # nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

        #LORMA
        
        self.A = nn.Parameter(
            torch.empty(rank, out_dim, device=device, dtype=dtype)
        )
        
        self.B = nn.Parameter(
            torch.zeros(out_dim, rank, device=device, dtype=dtype)
        )
        
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        #LORA
    # def forward(self, x):
    #     return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

        #LORMA
    def forward(self, x):
        y = self.base(x)
    
        # Apply the multiplicative transformation to W0*x,
        # excluding the frozen bias from the transformation.
        pre = y if self.base.bias is None else y - self.base.bias
    
        update = F.linear(
            F.linear(pre, self.A),
            self.B
        )
    
        return y + self.scale * update


def inject_low_rank(module, target_suffixes, rank, alpha):
    """Replace every nn.Linear whose name ends with a target suffix by LowRankMulLinear. Generic across backbones."""
    to_replace = [
        name for name, m in module.named_modules()
        if isinstance(m, nn.Linear)
        and any(name == t or name.endswith("." + t) for t in target_suffixes)
    ]
    for name in to_replace:
        parent_name, _, child = name.rpartition(".")
        parent = module.get_submodule(parent_name) if parent_name else module
        setattr(parent, child, LowRankMulLinear(getattr(parent, child), rank=rank, alpha=alpha))
    return to_replace

In [ ]:
ADAPTATION_METHODS = {}


def register_adaptation(name):
    def deco(fn):
        ADAPTATION_METHODS[name] = fn
        return fn
    return deco


# ------------------------------------------------------------
# Common helpers
# ------------------------------------------------------------

def _freeze_backbone(model):
    for p in model.backbone.parameters():
        p.requires_grad = False


def _freeze_all(model):
    for p in model.parameters():
        p.requires_grad = False


def _unfreeze_head(model):
    for name in model.head_module_names():
        module = getattr(model, name)

        for p in module.parameters():
            p.requires_grad = True


def _targets(cfg):
    t = BACKBONES[cfg.backbone_name]["adapter_targets"]

    if not t:
        raise ValueError(
            f"No adapter_targets registered for backbone "
            f"'{cfg.backbone_name}'"
        )

    return t


# ------------------------------------------------------------
# 1. Full Fine-Tuning
# ------------------------------------------------------------

@register_adaptation("full")
def apply_full(model, cfg):
    """
    Train every backbone parameter and retrieval-head parameter.
    """

    for p in model.parameters():
        p.requires_grad = True

    print("Full fine-tuning: all model parameters are trainable.")

    return model


# ------------------------------------------------------------
# 2. Partial Fine-Tuning
# ------------------------------------------------------------

@register_adaptation("partial")
def apply_partial(model, cfg):
    """
    Freeze the backbone initially.
    Unfreeze the last 25% of layers in each recognized ModuleList,
    with at least one layer selected.
    Keep the retrieval head trainable.

    Compatible with the notebook's decoder, SSM, enc-dec and CNN
    backbone layer containers.
    """

    _freeze_all(model)

    backbone = model.backbone

    layer_containers = []

    for module_name, module in backbone.named_modules():

        if isinstance(module, nn.ModuleList):

            n_layers = len(module)

            if n_layers == 0:
                continue

            # Unfreeze the final 25%, with a minimum of one layer.
            n_unfreeze = max(
                1,
                int(math.ceil(0.25 * n_layers))
            )

            layer_containers.append(
                (module_name, module, n_layers, n_unfreeze)
            )

    if not layer_containers:
        raise RuntimeError(
            "Partial fine-tuning could not identify any ModuleList "
            "layer containers in this backbone."
        )

    for name, container, n_layers, n_unfreeze in layer_containers:

        start_idx = n_layers - n_unfreeze

        for layer_idx in range(start_idx, n_layers):

            layer = container[layer_idx]

            for p in layer.parameters():
                p.requires_grad = True

        print(
            f"Partial FT: {name or '<root>'} | "
            f"total layers={n_layers} | "
            f"trainable layers={n_unfreeze}"
        )

    # Retrieval-specific head remains trainable.
    _unfreeze_head(model)

    return model


# ------------------------------------------------------------
# 3. IA3 (PEFT)
# ------------------------------------------------------------

class PeftBackboneShim(nn.Module):
    """
    Thin wrapper so PEFT does not inspect the Phase-1 model's
    plain dataclass config.
    """

    def __init__(self, model):
        super().__init__()
        self.model = model

    def encode(self, input_ids, attention_mask=None):
        return self.model.encode(input_ids, attention_mask)

    def forward(self, input_ids, attention_mask=None, **kwargs):
        return self.model.encode(input_ids, attention_mask)


def _wrap_with_peft(model, peft_cfg):
    model.backbone = get_peft_model(
        PeftBackboneShim(model.backbone),
        peft_cfg,
    )

    model.backbone.print_trainable_parameters()

    return model


@register_adaptation("ia3")
def apply_ia3(model, cfg):
    _freeze_backbone(model)

    info = BACKBONES[cfg.backbone_name]

    peft_cfg = IA3Config(
        target_modules=_targets(cfg),
        feedforward_modules=info["ia3_feedforward"],
    )

    _wrap_with_peft(model, peft_cfg)

    _unfreeze_head(model)

    return model


# ------------------------------------------------------------
# 4. LORMA (custom multiplicative low-rank injection)
# ------------------------------------------------------------

@register_adaptation("lorma")
def apply_lorma(model, cfg):
    _freeze_backbone(model)

    replaced = inject_low_rank(
        model.backbone,
        target_suffixes=_targets(cfg),
        rank=cfg.lorma_rank,
        alpha=cfg.lorma_alpha,
    )

    if not replaced:
        raise RuntimeError(
            f"LORMA: No matching linear layers found for "
            f"{_targets(cfg)}"
        )

    print(f"LORMA: wrapped {len(replaced)} Linear layers")

    _unfreeze_head(model)

    return model


# ------------------------------------------------------------
# Generic adaptation dispatcher
# ------------------------------------------------------------

def apply_adaptation(model, cfg):

    if cfg.adaptation_method not in ADAPTATION_METHODS:
        raise ValueError(
            f"Unknown adaptation '{cfg.adaptation_method}'. "
            f"Available: {list(ADAPTATION_METHODS)}"
        )

    model = ADAPTATION_METHODS[
        cfg.adaptation_method
    ](model, cfg)

    report_trainable(model)

    return model


def report_trainable(model):

    total = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    print(f"Total params    : {total:,}")

    print(
        f"Trainable params: {trainable:,} "
        f"({100 * trainable / max(total, 1):.3f}%)"
    )

    return total, trainable


# ------------------------------------------------------------
# Verify registered methods
# ------------------------------------------------------------

assert set(ADAPTATION_METHODS) == {
    "full",
    "partial",
    "ia3",
    "lorma",
}, ADAPTATION_METHODS.keys()

print("Adaptation methods:", list(ADAPTATION_METHODS))

## 11. Experiment configuration

`exp_cfg` is trained by section 14 when `RUN_TRAINING = True`. The active configuration is the KD teacher of the best
recorded run (decoder, full fine-tuning, lr 2e-5, attention token pooling). The commented configurations are other runs / methods.

In [ ]:
@dataclass
class ExperimentConfig:
    backbone_name: str
    adaptation_method: str  # "full" | "partial" | "ia3" | "lorma"

    learning_rate: float
    batch_size: int
    grad_accum: int
    epochs: int

    head_learning_rate: float = 3e-4
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_amp: bool = True
    dropout: float = 0.1

    pooling_strategy: str = "last_token"
    doc_pooling: str = "mean"

    # Retrieval
    embed_dim: int = 256
    contrastive_temperature: float = 0.05
    num_hard_negatives: int = NUM_HARD_NEGATIVES
    top_k: int = TOP_K
    exclude_self: bool = True

    # IA3
    # IA3 uses the registered backbone adapter targets.

    # LORMA
    lorma_rank: int = 16
    lorma_alpha: float = 32.0

    # Partial fine-tuning
    # Current implementation unfreezes the final 25% of ModuleList layers.

    seed: int = SEED
    run_name: Optional[str] = None

    @property
    def run_dir(self):
        name = self.run_name or (
            f"phase7_{self.backbone_name}_{self.adaptation_method}"
        )
        return WORK_DIR / name


def save_experiment_config(cfg, run_dir=None):
    run_dir = Path(run_dir or cfg.run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    with open(
        run_dir / "config.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(asdict(cfg), f, indent=2)


# ------------------------------------------------------------
# Active experiment
# ------------------------------------------------------------

# KD teacher of the best run (Codabench 0.1636, KD student): decoder, full fine-tuning, lr 2e-5.
exp_cfg = ExperimentConfig(
    backbone_name="decoder",
    adaptation_method="full",
    learning_rate=2e-5,
    batch_size=BATCH_SIZE,
    grad_accum=4,
    epochs=6,
    lorma_rank=16,
    lorma_alpha=32.0,
    pooling_strategy="attention",
)


# ------------------------------------------------------------
# Other experiment configurations
# ------------------------------------------------------------

# Full fine-tuning, lr 1e-5 (run "decoder_pairwise_hardneg_lr1e5")
# exp_cfg = ExperimentConfig(backbone_name="decoder", adaptation_method="full", learning_rate=1e-5,
#                            batch_size=BATCH_SIZE, grad_accum=4, epochs=6, lorma_rank=16, lorma_alpha=32.0,
#                            pooling_strategy="attention", run_name="decoder_pairwise_hardneg_lr1e5")

# Full fine-tuning
# exp_cfg = ExperimentConfig(
#     backbone_name="decoder",
#     adaptation_method="full",
#     learning_rate=2e-5,
#     batch_size=BATCH_SIZE,
#     grad_accum=4,
#     epochs=6,
#     pooling_strategy=DEFAULT_POOLING["decoder"],
# )

# Partial fine-tuning
# exp_cfg = ExperimentConfig(
#     backbone_name="ssm",
#     adaptation_method="partial",
#     learning_rate=5e-4,
#     batch_size=BATCH_SIZE,
#     grad_accum=4,
#     epochs=6,
#     pooling_strategy=DEFAULT_POOLING["ssm"],
# )

# IA3
# exp_cfg = ExperimentConfig(
#     backbone_name="encdec",
#     adaptation_method="ia3",
#     learning_rate=3e-3,
#     batch_size=BATCH_SIZE,
#     grad_accum=4,
#     epochs=6,
#     pooling_strategy=DEFAULT_POOLING["encdec"],
# )

# LORMA
# exp_cfg = ExperimentConfig(
#     backbone_name="cnn",
#     adaptation_method="lorma",
#     learning_rate=5e-4,
#     batch_size=BATCH_SIZE,
#     grad_accum=4,
#     epochs=6,
#     lorma_rank=16,
#     lorma_alpha=32.0,
#     pooling_strategy=DEFAULT_POOLING["cnn"],
# )


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert exp_cfg.top_k == 7

assert exp_cfg.backbone_name in BACKBONES, exp_cfg.backbone_name

assert exp_cfg.adaptation_method in ADAPTATION_METHODS

print(asdict(exp_cfg))

In [ ]:
def build_adapted_model(cfg, device=DEVICE, backbone=None, **extra_model_kwargs):
    """Phase-1 backbone + Phase-7 head + chosen adaptation. Use this both for training and for reloading checkpoints."""
    set_seed(cfg.seed)
    model = build_phase7_model(
        backbone_name=cfg.backbone_name,
        device="cpu",
        backbone=backbone,
        dropout=cfg.dropout,
        pooling_strategy=cfg.pooling_strategy,
        doc_pooling=cfg.doc_pooling,
        embed_dim=cfg.embed_dim,
        temperature=cfg.contrastive_temperature,
        **extra_model_kwargs,
    )
    model = apply_adaptation(model, cfg)
    return model.to(device)


def build_optimizer(model, cfg):
    head_names = tuple(f"{n}." for n in model.head_module_names())
    head_params, adapter_params = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (head_params if n.startswith(head_names) else adapter_params).append(p)

    groups = []
    if adapter_params:
        groups.append({"params": adapter_params, "lr": cfg.learning_rate})
    if head_params:
        groups.append({"params": head_params, "lr": cfg.head_learning_rate})
    return torch.optim.AdamW(groups, weight_decay=cfg.weight_decay)


def build_scheduler(optimizer, cfg, steps_per_epoch):
    total_steps = max(math.ceil(steps_per_epoch / cfg.grad_accum) * cfg.epochs, 1)
    return torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=[g["lr"] for g in optimizer.param_groups],
        total_steps=total_steps,
        pct_start=0.1,
        anneal_strategy="cos",
    )


def load_trained_model(cfg, checkpoint_path=None, backbone=None):
    """Rebuild backbone + retrieval head + adaptation for `cfg`, then load a trained Phase-7 checkpoint strictly."""
    path = Path(checkpoint_path or (cfg.run_dir / "best_model.pt"))
    if not path.is_file():
        raise FileNotFoundError(f"Trained checkpoint not found: {path}")
    model = build_adapted_model(cfg, backbone=backbone)
    model.load_state_dict(torch.load(path, map_location="cpu", weights_only=True), strict=True)
    model.eval()
    print("Loaded trained model:", path)
    return model

## 12. Training, evaluation and submission utilities

Definitions only — nothing in this section trains a model.

`train_one_epoch()` is the generic contrastive loop; `run_validation()` returns the dev contrastive loss. Ranking quality
(Micro-F1@7) is measured by ranking every dev query against the full dev pool with `run_inference()`.

In [ ]:
def split_batch(batch, device):
    tensors, meta = {}, {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            tensors[k] = v.to(device, non_blocking=True)
        else:
            meta[k] = v
    return tensors, meta


def _amp_enabled(use_amp):
    return use_amp and torch.cuda.is_available()


def make_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def train_one_epoch(
    model,
    loader,
    optimizer,
    device,
    grad_accum=1,
    max_grad_norm=1.0,
    use_amp=True,
    scaler=None,
    scheduler=None,
):
    model.train()
    amp = _amp_enabled(use_amp)

    if scaler is None:
        scaler = make_scaler(amp)

    assert grad_accum >= 1

    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    num_batches = len(loader)

    for step, batch in enumerate(tqdm(loader, desc="train", leave=False)):
        tensors, _ = split_batch(batch, device)

        remaining = num_batches - (step // grad_accum) * grad_accum
        current_accum = min(grad_accum, remaining)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=amp
        ):
            out = model(**tensors)
            raw_loss = out["loss"]
            loss = raw_loss / current_accum

        if not torch.isfinite(raw_loss):
            raise RuntimeError(
                f"Non-finite training loss at batch {step}"
            )

        scaler.scale(loss).backward()

        update_now = (
            (step + 1) % grad_accum == 0
            or (step + 1) == num_batches
        )

        if update_now:
            if max_grad_norm is not None:
                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad],
                    max_grad_norm
                )

            old_scale = scaler.get_scale()

            scaler.step(optimizer)
            scaler.update()

            step_succeeded = (
                not amp or scaler.get_scale() >= old_scale
            )

            optimizer.zero_grad(set_to_none=True)

            if scheduler is not None and step_succeeded:
                scheduler.step()

        total_loss += raw_loss.detach().item()

    return total_loss / max(num_batches, 1)


@torch.no_grad()
def run_validation(model, loader, device, use_amp=True):
    """Mean contrastive loss on dev batches (in-batch + hard negatives)."""
    model.eval()
    amp = _amp_enabled(use_amp)
    total_loss, n_loss = 0.0, 0
    for batch in tqdm(loader, desc="val-loss", leave=False):
        tensors, _ = split_batch(batch, device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            out = model(**tensors)
        if out.get("loss") is not None and torch.isfinite(out["loss"]):
            total_loss += out["loss"].item()
            n_loss += 1
    return {"loss": total_loss / n_loss if n_loss else None}

### 12.1 Metrics

Micro-averaged over all queries, using the **top 7** predicted IDs against the variable-length gold list:

- `hits = Σ_q |top7(q) ∩ gold(q)|`
- Micro-Precision@7 = `hits / Σ_q |top7(q)|`  (= `hits / (7 · #queries)`)
- Micro-Recall@7 = `hits / Σ_q |gold(q)|`
- Micro-F1@7 = `2 · P · R / (P + R)` (0 if P + R = 0)

In [ ]:
def compute_micro_f1_at_k(rankings, queries, k=TOP_K):
    """
    rankings : Dict[query_id, List[cand_id]]
    queries : List[{"id", "relevant"}]

    Returns micro-F1, precision, recall, and counts.
    """
    hits = 0
    num_predicted = 0
    num_gold = 0

    for q in queries:
        qid = q["id"]

        # Missing rankings must raise an error
        if qid not in rankings:
            raise KeyError(f"Missing ranking for query_id: {qid}")

        pred = rankings[qid][:k]
        gold = set(q["relevant"])

        hits += len(set(pred) & gold)
        num_predicted += len(pred)
        num_gold += len(gold)

    precision = (
        hits / num_predicted
        if num_predicted > 0
        else 0.0
    )

    recall = (
        hits / num_gold
        if num_gold > 0
        else 0.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    return {
        "micro_f1@7": f1,
        "micro_precision@7": precision,
        "micro_recall@7": recall,
        "hits": hits,
        "num_predicted": num_predicted,
        "num_gold": num_gold,
        "num_queries": len(queries),
    }

### 12.2 Ranking (inference) and submission helpers

Expected `prediction.json` — a JSON **dictionary**, every test query → exactly 7 unique candidate IDs from `test_candidates.json`, most relevant first:

```json
{
  "query_001": ["candA", "candB", "candC", "candD", "candE", "candF", "candG"],
  "query_002": ["..."]
}
```

In [ ]:
@torch.no_grad()
def run_inference(
    model,
    queries,
    pool,
    split_name,
    cfg,
    device=DEVICE,
    embed_batch=16,
):
    """
    Embed each query and candidate once, then rank candidates by cosine similarity.

    Returns:
        Dict[query_id, List[cand_id]] with exactly cfg.top_k unique IDs.
    """

    model.eval()

    cand_ids = list(pool.keys())

    if len(cand_ids) < cfg.top_k:
        raise ValueError(
            f"Candidate pool has {len(cand_ids)} cases, "
            f"but top_k={cfg.top_k}"
        )

    # --------------------------------------------------
    # Chunk selection policy: spread chunks across doc
    # --------------------------------------------------
    def select_chunks(chunks, max_chunks):
        if len(chunks) <= max_chunks:
            return chunks

        if max_chunks == 1:
            return [chunks[len(chunks) // 2]]

        indices = [
            round(i * (len(chunks) - 1) / (max_chunks - 1))
            for i in range(max_chunks)
        ]

        return [chunks[i] for i in indices]

    # --------------------------------------------------
    # Pad a batch of documents
    # --------------------------------------------------
    def pad_documents(documents):
        n_docs = len(documents)

        max_chunks = max(len(chunks) for chunks in documents)
        max_tokens = max(
            len(chunk)
            for chunks in documents
            for chunk in chunks
        )

        input_ids = torch.full(
            (n_docs, max_chunks, max_tokens),
            PAD_ID,
            dtype=torch.long,
        )

        attention_mask = torch.zeros(
            n_docs,
            max_chunks,
            max_tokens,
            dtype=torch.long,
        )

        chunk_mask = torch.zeros(
            n_docs,
            max_chunks,
            dtype=torch.float32,
        )

        for d, chunks in enumerate(documents):
            for c, chunk in enumerate(chunks):
                length = len(chunk)

                input_ids[d, c, :length] = torch.tensor(
                    chunk,
                    dtype=torch.long,
                )

                attention_mask[d, c, :length] = 1
                chunk_mask[d, c] = 1.0

        return input_ids, attention_mask, chunk_mask

    # --------------------------------------------------
    # Embed a list of documents in batches
    # --------------------------------------------------
    def encode_documents_in_batches(documents):
        embeddings = []

        for start in range(0, len(documents), embed_batch):
            batch_docs = documents[start:start + embed_batch]

            ids, mask, chunk_mask = pad_documents(batch_docs)

            ids = ids.to(device)
            mask = mask.to(device)
            chunk_mask = chunk_mask.to(device)

            batch_emb = model.encode_documents(
                ids,
                mask,
                chunk_mask,
            )

            embeddings.append(batch_emb.detach())

        return torch.cat(embeddings, dim=0)

    # --------------------------------------------------
    # 1. Prepare candidate documents
    # --------------------------------------------------
    candidate_chunks = []

    for cid in cand_ids:
        chunks = chunk_cache.get(
            (split_name, "pool", cid),
            pool[cid],
            max_chunks=None,
        )

        chunks = select_chunks(chunks, MAX_CAND_CHUNKS)

        candidate_chunks.append(chunks)

    # --------------------------------------------------
    # 2. Encode candidates once
    # --------------------------------------------------
    candidate_embeddings = encode_documents_in_batches(
        candidate_chunks
    )

    # --------------------------------------------------
    # 3. Prepare query documents
    # --------------------------------------------------
    query_ids = []
    query_chunks = []

    for q in queries:
        qid = str(q["id"])
        query_ids.append(qid)

        chunks = chunk_cache.get(
            (split_name, "query", qid),
            q["text"],
            max_chunks=None,
        )

        chunks = select_chunks(chunks, MAX_QUERY_CHUNKS)

        query_chunks.append(chunks)

    # --------------------------------------------------
    # 4. Encode queries once
    # --------------------------------------------------
    query_embeddings = encode_documents_in_batches(
        query_chunks
    )

    # --------------------------------------------------
    # 5. Score all queries against the candidate pool
    # --------------------------------------------------
    scores = query_embeddings @ candidate_embeddings.T

    # --------------------------------------------------
    # 6. Exclude self-retrieval if configured
    # --------------------------------------------------
    if cfg.exclude_self:
        candidate_id_to_idx = {
            cid: idx for idx, cid in enumerate(cand_ids)
        }

        for row, qid in enumerate(query_ids):
            if qid in candidate_id_to_idx:
                scores[row, candidate_id_to_idx[qid]] = -float("inf")

    # --------------------------------------------------
    # 7. Top-k rankings
    # --------------------------------------------------
    top_indices = scores.topk(
        cfg.top_k,
        dim=-1,
    ).indices

    rankings = {}

    for row, qid in enumerate(query_ids):
        ranked_ids = [
            cand_ids[idx]
            for idx in top_indices[row].tolist()
        ]

        rankings[qid] = ranked_ids

    return rankings


def validate_submission_format(
    predictions,
    test_queries,
    test_pool,
    k=TOP_K,
    exclude_self=False,
):
    if not isinstance(predictions, dict):
        raise TypeError("predictions must be a dict")

    query_ids = {str(q["id"]) for q in test_queries}

    if set(predictions.keys()) != query_ids:
        missing = query_ids - set(predictions.keys())
        extra = set(predictions.keys()) - query_ids

        raise ValueError(
            f"Query ID mismatch. Missing: {missing}, Extra: {extra}"
        )

    pool_ids = set(test_pool.keys())

    for q in test_queries:
        qid = str(q["id"])
        pred = predictions[qid]

        if not isinstance(pred, list):
            raise TypeError(f"Prediction for {qid} must be a list")

        if len(pred) != k:
            raise ValueError(
                f"Query {qid}: expected {k} predictions, "
                f"got {len(pred)}"
            )

        if not all(isinstance(cid, str) for cid in pred):
            raise TypeError(
                f"Query {qid}: all candidate IDs must be strings"
            )

        if len(set(pred)) != k:
            raise ValueError(
                f"Query {qid}: duplicate candidate IDs found"
            )

        invalid = set(pred) - pool_ids

        if invalid:
            raise ValueError(
                f"Query {qid}: candidates outside test pool: {invalid}"
            )

        if exclude_self and qid in pred:
            raise ValueError(
                f"Query {qid}: self-retrieval detected"
            )

    print("Submission format validation passed.")

In [ ]:
def save_submission(predictions, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    pred_path = output_dir / "prediction.json"
    zip_path = output_dir / "submission.zip"

    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(predictions, f, ensure_ascii=False, indent=2)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pred_path, arcname="prediction.json")

    print("Saved:", pred_path)
    print("Saved:", zip_path)
    return pred_path, zip_path


def verify_zip_root(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        assert names == ["prediction.json"], f"zip must contain only prediction.json at root, got {names}"
        obj = json.loads(z.read("prediction.json").decode("utf-8"))
    print("Zip root OK:", names, "| entries:", len(obj))
    return obj

### 12.3 Training loop

In [ ]:
def fit(cfg, train_loader, dev_loader):
    run_dir = cfg.run_dir
    run_dir.mkdir(parents=True, exist_ok=True)
    save_experiment_config(cfg)

    model = build_adapted_model(cfg)
    optimizer = build_optimizer(model, cfg)
    scheduler = build_scheduler(optimizer, cfg, len(train_loader))
    scaler = make_scaler(_amp_enabled(cfg.use_amp))

    best_f1, history = -1.0, []

    for epoch in range(cfg.epochs):
        t0 = time.time()
        train_loss = train_one_epoch(
            model, train_loader, optimizer, DEVICE,
            grad_accum=cfg.grad_accum, max_grad_norm=cfg.max_grad_norm,
            use_amp=cfg.use_amp, scaler=scaler, scheduler=scheduler,
        )
        val_loss = run_validation(model, dev_loader, DEVICE, use_amp=cfg.use_amp)["loss"]
        dev_rankings = run_inference(model, dev_queries, dev_pool, "dev", cfg)            # Section 12
        metrics = compute_micro_f1_at_k(dev_rankings, dev_queries, k=cfg.top_k)          # Section 10

        row = {"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss,
               **{k: metrics[k] for k in ("micro_f1@7", "micro_precision@7", "micro_recall@7")},
               "time_s": round(time.time() - t0, 1)}
        history.append(row)
        print(row)

        if metrics["micro_f1@7"] > best_f1:
            best_f1 = metrics["micro_f1@7"]
            torch.save(model.state_dict(), run_dir / "best_model.pt")
            with open(run_dir / "metrics.json", "w") as f:
                json.dump(metrics, f, indent=2)
            print(f"  ✓ new best Micro-F1@7 {best_f1:.4f}")

    with open(run_dir / "history.json", "w") as f:
        json.dump(history, f, indent=2)

    # reload the best checkpoint (as in Phase 3) so the returned model is the selected one
    model.load_state_dict(torch.load(run_dir / "best_model.pt", map_location=DEVICE, weights_only=True))
    print(f"Training complete. Best: {best_f1:.4f}")
    return model, best_f1, history

In [ ]:
def tiny_overfit_test(cfg, n_items=4, steps=30):
    model = build_adapted_model(cfg)

    tiny_loader = DataLoader(
        Subset(train_ds, range(min(n_items, len(train_ds)))),
        batch_size=1,
        shuffle=True,
        collate_fn=pcr_collator,
    )

    opt = build_optimizer(
        model,
        replace(cfg, learning_rate=cfg.learning_rate * 5)
    )

    scaler = make_scaler(_amp_enabled(cfg.use_amp))

    for i in range(steps):
        loss = train_one_epoch(
            model,
            tiny_loader,
            opt,
            DEVICE,
            grad_accum=1,
            use_amp=cfg.use_amp,
            scaler=scaler,
        )

        if i % 5 == 0 or i == steps - 1:
            print(f"step {i:3d} loss {loss:.4f}")

    del model, opt, scaler, tiny_loader
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

### 12.4 Knowledge-distillation utilities

Definitions used by section 15 (and the KD-loss preflight check).

In [ ]:
class DistillationExperiment:
    """
    Generic teacher -> student distillation (same structure as Phase 3, task-specific KD term plugged in).
      kd_loss_fn(student_out, teacher_out, tensors, temperature) -> scalar soft-target loss
    Total loss = (1 - alpha) * gold_loss(student) + alpha * kd_loss
    """

    def __init__(self, teacher, student, kd_loss_fn, temperature=2.0, alpha=0.5, teacher_drop_keys=()):
        self.teacher = teacher
        self.student = student
        self.kd_loss_fn = kd_loss_fn
        self.temperature = temperature
        self.alpha = alpha
        self.teacher_drop_keys = set(teacher_drop_keys)   # e.g. gold labels: the teacher needs no gold loss

        self.teacher.eval()
        for p in self.teacher.parameters():
            p.requires_grad = False

    def compute_loss(self, tensors):
        with torch.no_grad():
            teacher_out = self.teacher(**{k: v for k, v in tensors.items() if k not in self.teacher_drop_keys})

        student_out = self.student(**tensors)
        gold_loss = student_out["loss"]
        kd_loss = self.kd_loss_fn(student_out, teacher_out, tensors, self.temperature)

        loss = (1.0 - self.alpha) * gold_loss + self.alpha * kd_loss
        return {"loss": loss, "gold_loss": gold_loss, "kd_loss": kd_loss}


def train_kd_epoch(
    kd_exp,
    loader,
    optimizer,
    device,
    grad_accum=1,
    max_grad_norm=1.0,
    use_amp=True,
    scaler=None,
    scheduler=None,
):
    kd_exp.student.train()
    kd_exp.teacher.eval()
    amp = _amp_enabled(use_amp)
    if scaler is None:
        scaler = make_scaler(amp)

    optimizer.zero_grad(set_to_none=True)
    tot = tot_gold = tot_kd = 0.0
    num_batches = len(loader)

    for step, batch in enumerate(tqdm(loader, desc="kd", leave=False)):
        tensors, _ = split_batch(batch, device)

        remaining = num_batches - (step // grad_accum) * grad_accum
        current_accum = min(grad_accum, remaining)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            out = kd_exp.compute_loss(tensors)
            loss = out["loss"] / current_accum

        if not torch.isfinite(out["loss"]):
            raise RuntimeError(f"Non-finite KD loss at batch {step}")

        scaler.scale(loss).backward()

        if (step + 1) % grad_accum == 0 or (step + 1) == num_batches:
            if max_grad_norm is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in kd_exp.student.parameters() if p.requires_grad], max_grad_norm
                )
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            step_succeeded = not amp or scaler.get_scale() >= old_scale
            optimizer.zero_grad(set_to_none=True)
            if scheduler is not None and step_succeeded:
                scheduler.step()

        tot += out["loss"].item()
        tot_gold += out["gold_loss"].item()
        tot_kd += out["kd_loss"].item()

    n = max(num_batches, 1)
    return tot / n, tot_gold / n, tot_kd / n


def build_truncated_decoder_backbone(n_layers):
    """Smaller student backbone (as in Phase 3): Phase-1 decoder keeping embeddings + first `n_layers` blocks + final_ln."""
    student_config = replace(decoder_config, n_layer=n_layers)
    backbone = DecoderOnlyTransformer(student_config)
    res = backbone.load_state_dict(_load_state(DECODER_CHECKPOINT), strict=False)
    assert not res.missing_keys, res.missing_keys
    assert all(k.startswith("blocks.") and int(k.split(".")[1]) >= n_layers for k in res.unexpected_keys), \
        [k for k in res.unexpected_keys if not k.startswith("blocks.")][:5]
    print(f"Student decoder: {n_layers} layers | dropped tensors from deeper layers: {len(res.unexpected_keys)}")
    return backbone


def listwise_kd_loss(student_out, teacher_out, tensors, T):
    s, t = student_out["scores"].float(), teacher_out["scores"].float()      # [B, M]
    kl = F.kl_div(F.log_softmax(s / T, dim=-1), F.softmax(t / T, dim=-1), reduction="batchmean")
    return kl * (T * T)

## 13. Preflight tests (optional, lightweight)

Run when `RUN_PREFLIGHT = True`. No full training; ranking checks use a small slice of the test pool. Every check is reported
as PASS/FAIL; a missing checkpoint is reported with its exact path (never skipped), and the summary cell raises if anything failed.

1. Every backbone in `PREFLIGHT_BACKBONES` loads strictly from its checkpoint.
2. Every `encode()` returns `[B, L, D]`.
3. Dataset and collator: tensor shapes, ≥ 1 positive per query, candidates from the query's own pool, no self-candidate, de-duplicated candidates.
4. The bi-encoder returns `[B, M]` scores and a finite loss for every token-pooling strategy; embeddings are unit-norm.
5. Padded chunk slots do not change a document's embedding.
6. Every adaptation method trains exactly the intended parameters (all backbones).
7. A small batch supports forward, loss and backward with gradients only on trainable parameters.
8. Scoring without relevance labels works (`loss` is `None`).
9. Micro-F1@7 and the listwise KD loss behave as expected on known inputs.
10. Ranking returns 7 unique in-pool candidates per query (self excluded); submission files validate.

In [ ]:
PREFLIGHT_RESULTS = []


def preflight_check(name, fn):
    try:
        detail = fn()
        PREFLIGHT_RESULTS.append((name, "PASS", detail or ""))
        print(f"[PASS] {name}" + (f" - {detail}" if detail else ""))
    except Exception as exc:
        PREFLIGHT_RESULTS.append((name, "FAIL", f"{type(exc).__name__}: {exc}"))
        print(f"[FAIL] {name} - {type(exc).__name__}: {exc}")


def _fail(msg):
    raise RuntimeError(msg)


def small_pcr_batch(n_tok=24):
    """One training item with its chunks shortened to n_tok tokens (keeps preflight cheap)."""
    item = train_ds[0]
    cut = lambda chunks: [c if len(c) <= n_tok else c[: n_tok - 1] + [EOS_ID] for c in chunks]
    item = dict(item, query_chunks=cut(item["query_chunks"]), cand_chunks=[cut(c) for c in item["cand_chunks"]])
    return pcr_collator([item])


def is_adapter_param(name):
    return "ia3_" in name or name.endswith(".A") or name.endswith(".B")

In [ ]:
# ---- Tests 1, 2, 6: every backbone -> strict load, encode() shape, every adaptation method ----
pf_backbone = None

if RUN_PREFLIGHT:
    probe_ids = torch.tensor([[BOS_ID] + tokenizer.encode("The appeal is dismissed with costs.") + [EOS_ID]] * 2)

    for name in PREFLIGHT_BACKBONES:
        info = BACKBONES[name]
        ckpt = info["checkpoint_path"]
        loaded = {}

        def _load():
            if not Path(ckpt).exists():
                raise FileNotFoundError(f"missing checkpoint: {ckpt}")
            bb = load_backbone(name, device="cpu", strict=True)
            loaded["bb"] = bb
            return f"strict load OK ({sum(p.numel() for p in bb.parameters()):,} params)"

        preflight_check(f"1. {name}: checkpoint loads strictly", _load)
        if "bb" not in loaded:
            preflight_check(f"2. {name}: encode() shape", lambda: _fail("backbone not loaded"))
            continue
        bb = loaded["bb"]

        def _encode():
            bb.eval()
            with torch.no_grad():
                h = bb.encode(probe_ids, torch.ones_like(probe_ids))
            assert tuple(h.shape) == (2, probe_ids.shape[1], info["hidden_dim"]), tuple(h.shape)
            assert CHUNK_TOKENS <= BACKBONE_MAX_TOKENS[name]
            return f"{tuple(h.shape)}"

        preflight_check(f"2. {name}: encode() returns [B, L, D]", _encode)

        for method in ["full", "partial", "ia3", "lorma"]:
            def _adapt(method=method):
                cfg = replace(exp_cfg, backbone_name=name, adaptation_method=method)
                m = Phase7PCRModel(copy.deepcopy(bb), info["hidden_dim"], pooling_strategy=exp_cfg.pooling_strategy)
                m = ADAPTATION_METHODS[method](m, cfg)
                head = tuple(f"{h}." for h in m.head_module_names())
                head_frozen = [n for n, p in m.named_parameters() if n.startswith(head) and not p.requires_grad]
                assert not head_frozen, f"frozen head parameters: {head_frozen[:3]}"
                bb_params = list(m.backbone.named_parameters())
                trainable = [n for n, p in bb_params if p.requires_grad]
                if method == "full":
                    assert len(trainable) == len(bb_params), "full fine-tuning left backbone parameters frozen"
                elif method == "partial":
                    assert 0 < len(trainable) < len(bb_params), "partial fine-tuning must train some, not all, layers"
                else:
                    wrong = [n for n in trainable if not is_adapter_param(n)]
                    assert trainable and not wrong, f"unexpected trainable backbone parameters: {wrong[:3]}"
                n_tr = sum(p.numel() for _, p in bb_params if p.requires_grad)
                del m
                return f"trainable backbone params {n_tr:,}"

            preflight_check(f"6. {name}: {method} trainable parameters", _adapt)

        if name == exp_cfg.backbone_name:
            pf_backbone = bb          # reused below; avoids loading the checkpoint again
        else:
            del bb
        loaded.clear()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
else:
    print("RUN_PREFLIGHT=False: preflight skipped.")

In [ ]:
# ---- Tests 3, 4, 5, 7, 8, 9 ----
if RUN_PREFLIGHT:

    def _collator():
        item = train_ds[0]
        assert item["positive_ids"], "query without positives"
        assert all(c in train_pool for c in item["cand_ids"]), "candidate outside the query's pool"
        assert item["query_id"] not in item["cand_ids"], "query is its own candidate"
        b = pcr_collator([train_ds[0], train_ds[1]])
        B, Cq, L = b["query_input_ids"].shape
        M = b["cand_input_ids"].shape[0]
        assert b["relevance"].shape == (B, M) and (b["relevance"].sum(1) >= 1).all()
        assert len(set(b["candidate_ids"])) == M, "candidates not de-duplicated"
        assert Cq <= MAX_QUERY_CHUNKS and b["cand_input_ids"].shape[1] <= MAX_CAND_CHUNKS and L <= CHUNK_TOKENS
        assert (b["cand_attention_mask"].sum(-1) >= 1).all() and (b["query_attention_mask"].sum(-1) >= 1).all()
        return f"queries {tuple(b['query_input_ids'].shape)}, candidates {tuple(b['cand_input_ids'].shape)}, relevance {(B, M)}"

    preflight_check("3. dataset / collator", _collator)

    def _metrics_and_kd():
        qs = [{"id": "q1", "relevant": ["a", "b"]}, {"id": "q2", "relevant": ["c"]}]
        r = {"q1": ["a", "x1", "x2", "x3", "x4", "x5", "x6"], "q2": ["c", "y1", "y2", "y3", "y4", "y5", "y6"]}
        m = compute_micro_f1_at_k(r, qs)
        assert m["hits"] == 2 and abs(m["micro_precision@7"] - 2 / 14) < 1e-9 and abs(m["micro_recall@7"] - 2 / 3) < 1e-9
        s1 = torch.randn(2, 5)
        same = listwise_kd_loss({"scores": s1}, {"scores": s1}, {}, 2.0).item()
        diff = listwise_kd_loss({"scores": s1}, {"scores": torch.randn(2, 5)}, {}, 2.0).item()
        assert abs(same) < 1e-5 and diff > 1e-4, (same, diff)
        return f"Micro-F1@7 {m['micro_f1@7']:.4f}; KD identical {same:.1e}, different {diff:.3f}"

    preflight_check("9. Micro-F1@7 and listwise KD loss", _metrics_and_kd)

    if pf_backbone is None:
        preflight_check("4-8. model tests", lambda: _fail(f"backbone '{exp_cfg.backbone_name}' was not loaded in the previous cell"))
    else:
        hidden = BACKBONES[exp_cfg.backbone_name]["hidden_dim"]
        pb = small_pcr_batch()
        pt = {k: v for k, v in pb.items() if torch.is_tensor(v)}
        B, M = pt["relevance"].shape

        def _forward():
            done = []
            for pooling in POOLING_STRATEGIES:
                m = Phase7PCRModel(pf_backbone, hidden, pooling_strategy=pooling).eval()
                with torch.no_grad():
                    out = m(**pt)
                    emb = m.encode_documents(pt["cand_input_ids"], pt["cand_attention_mask"], pt["cand_chunk_mask"])
                assert tuple(out["scores"].shape) == (B, M) and torch.isfinite(out["loss"])
                assert torch.allclose(emb.norm(dim=-1), torch.ones(M), atol=1e-4)
                done.append(pooling)
            return f"{', '.join(done)} -> scores {(B, M)}, unit-norm embeddings"

        preflight_check("4. bi-encoder forward", _forward)

        def _padding():
            m = Phase7PCRModel(pf_backbone, hidden, pooling_strategy=exp_cfg.pooling_strategy).eval()
            ids, mask, cmask = pt["cand_input_ids"], pt["cand_attention_mask"], pt["cand_chunk_mask"]
            # pad the first candidate with an extra empty chunk slot and longer token rows
            pad_ids = torch.full((1, ids.shape[1] + 1, ids.shape[2] + 5), PAD_ID, dtype=torch.long)
            pad_mask = torch.zeros_like(pad_ids)
            pad_cmask = torch.zeros(1, ids.shape[1] + 1)
            pad_ids[0, : ids.shape[1], : ids.shape[2]] = ids[0]
            pad_mask[0, : ids.shape[1], : ids.shape[2]] = mask[0]
            pad_cmask[0, : ids.shape[1]] = cmask[0]
            pad_ids[0, -1, 0] = BOS_ID
            pad_mask[0, -1, 0] = 1
            with torch.no_grad():
                a = m.encode_documents(ids[:1], mask[:1], cmask[:1])
                b = m.encode_documents(pad_ids, pad_mask, pad_cmask)
            diff = (a - b).abs().max().item()
            assert diff < 1e-4, f"padding changed the embedding by {diff:.2e}"
            return f"max diff {diff:.1e}"

        preflight_check("5. padded chunks do not change embeddings", _padding)

        def _backward():
            m = build_adapted_model(exp_cfg, device=DEVICE, backbone=copy.deepcopy(pf_backbone)).train()
            out = m(**{k: v.to(DEVICE) for k, v in pt.items()})
            out["loss"].backward()
            no_grad = [n for n, p in m.named_parameters() if p.requires_grad and p.grad is None
                       and not n.startswith(("chunk_pool", "doc_pool"))]
            leaked = [n for n, p in m.named_parameters() if not p.requires_grad and p.grad is not None]
            assert not no_grad, f"trainable parameters without gradient: {no_grad[:3]}"
            assert not leaked, f"frozen parameters received gradients: {leaked[:3]}"
            n = sum(1 for p in m.parameters() if p.requires_grad)
            del m, out
            gc.collect()
            return f"{exp_cfg.adaptation_method}: {n} trainable tensors, gradients reach all used parameters"

        preflight_check("7. forward + loss + backward on a small batch", _backward)

        def _no_labels():
            m = Phase7PCRModel(pf_backbone, hidden, pooling_strategy=exp_cfg.pooling_strategy).eval()
            with torch.no_grad():
                out = m(**{k: v for k, v in pt.items() if k != "relevance"})
            assert out["loss"] is None and tuple(out["scores"].shape) == (B, M)
            return "loss=None, scores OK"

        preflight_check("8. scoring without relevance labels", _no_labels)

In [ ]:
# ---- Test 10: ranking + submission files on a slice of the test split (hash-embedding model, no backbone) ----
if RUN_PREFLIGHT:

    class _HashEmbedder(nn.Module):
        """Deterministic pseudo-embeddings from the token ids (no backbone)."""
        def encode_documents(self, input_ids, attention_mask, chunk_mask):
            g = torch.Generator().manual_seed(0)
            table = torch.randn(tokenizer.vocab_size, 16, generator=g)
            e = (table[input_ids.cpu()] * attention_mask.cpu().unsqueeze(-1)).sum((1, 2))
            return F.normalize(e, dim=-1)

    def _ranking():
        qs = test_queries[:5]
        keep = {q["id"] for q in qs} | set(list(test_pool)[:30])
        pool = {k: v for k, v in test_pool.items() if k in keep}
        cfg = replace(exp_cfg, run_name="_preflight")
        preds = run_inference(_HashEmbedder(), qs, pool, "test", cfg, device="cpu")
        validate_submission_format(preds, qs, pool, exclude_self=cfg.exclude_self)
        assert list(preds) == [q["id"] for q in qs]
        _, zip_path = save_submission(preds, WORK_DIR / "_preflight_submission")
        verify_zip_root(zip_path)
        return f"{len(preds)} queries ranked against {len(pool)} candidates, 7 unique each, self excluded"

    preflight_check("10. ranking and submission validation", _ranking)

In [ ]:
# ---- Preflight summary ----
if RUN_PREFLIGHT:
    pf_backbone = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    failed = [r for r in PREFLIGHT_RESULTS if r[1] == "FAIL"]
    print(f"\nPreflight: {len(PREFLIGHT_RESULTS) - len(failed)} passed, {len(failed)} failed")
    for name, status, detail in PREFLIGHT_RESULTS:
        print(f"  {status}  {name}" + (f"  ({detail})" if status == "FAIL" else ""))
    if failed:
        raise RuntimeError("Preflight failed: " + "; ".join(f"{n}: {d}" for n, _, d in failed))

### 13.1 Tiny overfit test (optional)

Train on a handful of queries; the contrastive loss should drop sharply.

In [ ]:
if RUN_TINY_OVERFIT:
    tiny_overfit_test(exp_cfg, n_items=4, steps=30)
else:
    print("RUN_TINY_OVERFIT=False: skipped.")

## 14. Main training experiment

Trains **only** `exp_cfg` (section 11) when `RUN_TRAINING = True`. Model selection: dev Micro-F1@7 from ranking the full dev
pool after every epoch. Saves `best_model.pt`, `metrics.json`, `history.json` and `config.json` in `exp_cfg.run_dir`, and
reloads the best checkpoint.

In [ ]:
model = None

if RUN_TRAINING:
    model, best_f1, history = fit(exp_cfg, train_loader, dev_loader)
else:
    print("RUN_TRAINING=False: training skipped.")

## 15. Knowledge distillation (optional)

`teacher (trained Phase-7 retriever) → relevance scores over the batch's candidates → smaller student`

Listwise KD per query: `T² · KL(softmax(t_scores/T) ‖ softmax(s_scores/T))`; total = `(1 − α) · InfoNCE(student) + α · KD`.
(Alternatives: pairwise margin-MSE on (positive, negative) score differences, or embedding-level distillation.)

Runs only when `RUN_KD = True`. Teacher: decoder + full fine-tuning (`teacher_cfg`), loaded strictly from
`KD_TEACHER_CHECKPOINT` or, if that is `None`, from the checkpoint trained in section 14. Student: the first `STUDENT_LAYERS`
blocks of the Phase-1 decoder with LORMA and the teacher's pooling.

In [ ]:
KD_TEMPERATURE = 2.0
KD_ALPHA = 0.5
KD_LR = 5e-4
KD_EPOCHS = 6
STUDENT_LAYERS = 4

# ------------------------------------------------------------
# Teacher: your best decoder full fine-tuning experiment
# ------------------------------------------------------------

# Same settings as the best run; run_dir = phase7_decoder_full (where section 14 saves it).
teacher_cfg = ExperimentConfig(
    backbone_name="decoder",
    adaptation_method="full",
    learning_rate=2e-5,
    batch_size=BATCH_SIZE,
    grad_accum=4,
    epochs=6,
    lorma_rank=16,
    lorma_alpha=32.0,
    pooling_strategy="attention",
)
TEACHER_PATH = Path(KD_TEACHER_CHECKPOINT) if KD_TEACHER_CHECKPOINT else teacher_cfg.run_dir / "best_model.pt"
print("Teacher checkpoint:", TEACHER_PATH)

# ------------------------------------------------------------
# Student: truncated decoder + LORMA
# ------------------------------------------------------------

student_cfg = replace(
    teacher_cfg,
    backbone_name="decoder",
    adaptation_method="lorma",
    pooling_strategy=teacher_cfg.pooling_strategy,
    doc_pooling=teacher_cfg.doc_pooling,
    run_name="decoder_4layer_lorma_kd",
    learning_rate=KD_LR,
    epochs=KD_EPOCHS,
)

assert student_cfg.backbone_name == "decoder"

print("Teacher run:", teacher_cfg.run_dir.name)
print("Student:", student_cfg.run_name)
print("Student layers:", STUDENT_LAYERS)
print("KD temperature:", KD_TEMPERATURE)
print("KD alpha:", KD_ALPHA)


def build_kd_teacher(checkpoint_path=None):
    path = Path(checkpoint_path or TEACHER_PATH)
    if not path.is_file():
        raise FileNotFoundError(
            f"KD teacher checkpoint not found: {path}. Train exp_cfg (section 14) or set KD_TEACHER_CHECKPOINT."
        )
    teacher = build_adapted_model(teacher_cfg)
    teacher.load_state_dict(torch.load(path, map_location="cpu", weights_only=True), strict=True)
    teacher.eval()
    print("Teacher loaded:", path)
    return teacher


def build_kd_student():
    # truncated Phase-1 decoder (embeddings + first STUDENT_LAYERS blocks + final_ln) + LORMA
    return build_adapted_model(student_cfg, backbone=build_truncated_decoder_backbone(STUDENT_LAYERS))


def build_kd_pair():
    teacher = build_kd_teacher()     # fails early if the teacher checkpoint is missing
    student = build_kd_student()
    return teacher, student


def run_kd():
    teacher, student = build_kd_pair()
    kd_exp = DistillationExperiment(teacher, student, listwise_kd_loss,
                                    temperature=KD_TEMPERATURE, alpha=KD_ALPHA, teacher_drop_keys=("relevance",))
    optimizer = build_optimizer(student, student_cfg)
    scheduler = build_scheduler(optimizer, student_cfg, len(train_loader))

    run_dir = student_cfg.run_dir
    run_dir.mkdir(parents=True, exist_ok=True)
    save_experiment_config(student_cfg)
    best_f1 = -1.0

    for epoch in range(student_cfg.epochs):
        loss, gold, kd = train_kd_epoch(kd_exp, train_loader, optimizer, DEVICE,
                                        grad_accum=student_cfg.grad_accum, use_amp=student_cfg.use_amp,
                                        scheduler=scheduler)
        dev_rankings = run_inference(student, dev_queries, dev_pool, "dev", student_cfg)
        metrics = compute_micro_f1_at_k(dev_rankings, dev_queries, k=student_cfg.top_k)
        print(f"epoch {epoch+1}: loss={loss:.4f} gold={gold:.4f} kd={kd:.4f} Micro-F1@7={metrics['micro_f1@7']:.4f}")

        if metrics["micro_f1@7"] > best_f1:
            best_f1 = metrics["micro_f1@7"]
            torch.save(student.state_dict(), run_dir / "best_model.pt")
            with open(run_dir / "metrics.json", "w") as f:
                json.dump({**metrics, "temperature": KD_TEMPERATURE, "alpha": KD_ALPHA,
                           "student_layers": STUDENT_LAYERS, "teacher": teacher_cfg.run_name}, f, indent=2)
    student.load_state_dict(torch.load(run_dir / "best_model.pt", map_location=DEVICE, weights_only=True))
    return student, best_f1


def load_kd_student(checkpoint_path=None):
    path = Path(checkpoint_path or (student_cfg.run_dir / "best_model.pt"))
    if not path.is_file():
        raise FileNotFoundError(f"KD student checkpoint not found: {path}")
    student = build_kd_student()
    student.load_state_dict(torch.load(path, map_location="cpu", weights_only=True), strict=True)
    student.eval()
    print("Loaded KD student:", path)
    return student


def kd_efficiency_report(student):
    teacher = build_kd_teacher()
    t_total = sum(p.numel() for p in teacher.parameters())
    s_total = sum(p.numel() for p in student.parameters())
    s_train = sum(p.numel() for p in student.parameters() if p.requires_grad)
    del teacher
    gc.collect()
    report = {"teacher_total_params": t_total, "student_total_params": s_total,
              "student_trainable_params": s_train, "param_reduction_pct": round(100 * (1 - s_total / t_total), 2)}
    print(json.dumps(report, indent=2))
    return report

### 15.1 Run knowledge distillation

In [ ]:
kd_student = None

if RUN_KD:
    kd_student, kd_best_f1 = run_kd()
    print("KD Best Micro-F1@7:", kd_best_f1)
    kd_metadata = {"teacher_checkpoint": str(TEACHER_PATH), "student_layers": STUDENT_LAYERS,
                   "temperature": KD_TEMPERATURE, "alpha": KD_ALPHA, "best_dev_micro_f1@7": kd_best_f1,
                   **kd_efficiency_report(kd_student)}
    with open(student_cfg.run_dir / "kd_metadata.json", "w", encoding="utf-8") as f:
        json.dump(kd_metadata, f, indent=2)
else:
    print("RUN_KD=False: knowledge distillation skipped.")

## 16. Test inference (ranking)

`INFERENCE_SOURCE = "main"` uses the `exp_cfg` model (trained above, or loaded from `INFERENCE_CHECKPOINT` /
`exp_cfg.run_dir/best_model.pt`); `"kd"` uses the KD student. The whole test pool is embedded once and every test query
gets its 7 most similar candidates (self excluded). Each source writes to its own run directory, so the main and KD
submissions cannot be mixed up.

In [ ]:
if RUN_INFERENCE:
    if INFERENCE_SOURCE == "main":
        inference_cfg = exp_cfg
        inference_model = model if model is not None else load_trained_model(exp_cfg, INFERENCE_CHECKPOINT)
    elif INFERENCE_SOURCE == "kd":
        inference_cfg = student_cfg
        inference_model = kd_student if kd_student is not None else load_kd_student(INFERENCE_CHECKPOINT)
    else:
        raise ValueError(f"INFERENCE_SOURCE must be 'main' or 'kd', got {INFERENCE_SOURCE!r}")

    inference_model = inference_model.to(DEVICE).eval()
    print("Inference source:", INFERENCE_SOURCE, "| run:", inference_cfg.run_dir.name)

    test_predictions = run_inference(inference_model, test_queries, test_pool, "test", inference_cfg)
else:
    print("RUN_INFERENCE=False: test inference skipped.")

## 17. Submission validation and ZIP

Expected `prediction.json` — a JSON **dictionary**, every test query → exactly 7 unique candidate IDs from `test_candidates.json`, most relevant first:

```json
{
  "query_001": ["candA", "candB", "candC", "candD", "candE", "candF", "candG"],
  "query_002": ["..."]
}
```

In [ ]:
if RUN_INFERENCE:
    validate_submission_format(test_predictions, test_queries, test_pool, exclude_self=inference_cfg.exclude_self)

    pred_path, zip_path = save_submission(test_predictions, inference_cfg.run_dir)
    verify_zip_root(zip_path)
    print("Final submission ready:", zip_path)
else:
    print("RUN_INFERENCE=False: no submission written.")

## Recorded Codabench results (Phase 7)

| Run | Backbone | Adaptation | Codabench Micro-F1@7 |
|---|---|---|---|
| kddecfull | decoder, 4-layer student | KD (teacher: decoder full FT, lr 2e-5) + LORMA | **0.1636** (best) |
| decfulllr2e-5 | decoder | different lr | 0.1636 |
| submission_5 | — | — | 0.1636 |
| cnnlormaattn | cnn | LORMA, attention pooling | 0.1258 |
| cnnlrchange | cnn | LORMA, different learning rate | 0.1181 |
| cnnlormamean | cnn | LORMA, mean pooling | 0.0963 |
